# CRL focus explorer

1. Run **1 · Prepare XPOSim**.
2. Run **2 · CRL and source parameters**. Enter the source, CRL geometry and material first. The element section is next to its geometry fields. The optical layout and analytical estimates follow the inputs.
3. Run **3 · Results**, then press **Compute**. No optical calculation starts before that click. To change the lens or source later, edit the fields in cell 2 and press **Compute** again in cell 3.

After a successful calculation, use **Download PDF report** at the bottom of Results for a one-page A4 report with the optical layout, scan and transverse examples before, at and after the numerical peak reference.

**Run all** initializes both panels without calculating. For GPU: **Runtime → Change runtime type → GPU**, reconnect, and run all. The actual CUDA device is checked.


In [ ]:
#@title 1 · Prepare XPOSim
import hashlib
import importlib.util
import importlib.metadata
import re
import subprocess
import sys
import urllib.request
from pathlib import Path

SOURCE_REF = '9909d09931938011fbc9b726b66bea08ce90caa9'
SOURCE_SHA256 = '5a5ec07e5a49f2a778fcd5d36a39eaf29661c3c32ec539ce6a536850a8af144c'
PROJECT_DIR = Path.cwd() / '.xposim_runtime' / SOURCE_REF
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
requirements = {'numpy':'numpy', 'scipy':'scipy', 'matplotlib':'matplotlib', 'ipywidgets':'ipywidgets>=8,<9'}
missing = [requirement for name,requirement in requirements.items() if importlib.util.find_spec(name) is None]
if importlib.util.find_spec('ipywidgets') is not None:
    try:
        widget_major = int(importlib.metadata.version('ipywidgets').split('.')[0])
    except (importlib.metadata.PackageNotFoundError, ValueError):
        widget_major = None
    if widget_major != 8:
        missing.append(requirements['ipywidgets'])
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet','--no-cache-dir',*missing])
try:
    gpu_info = subprocess.run(['nvidia-smi'],capture_output=True,text=True,timeout=15)
except (FileNotFoundError,subprocess.TimeoutExpired):
    gpu_info = None
if gpu_info is not None and gpu_info.returncode==0 and importlib.util.find_spec('cupy') is None:
    try:
        toolkit = subprocess.run(['nvcc','--version'],capture_output=True,text=True,timeout=15)
        release = re.search(r'release\s+(\d+)\.',toolkit.stdout)
    except (FileNotFoundError,subprocess.TimeoutExpired):
        release = None
    if release is None:
        release = re.search(r'CUDA Version:\s*(\d+)\.',gpu_info.stdout)
    major = int(release.group(1)) if release else None
    wheel = {12:'cupy-cuda12x',13:'cupy-cuda13x'}.get(major)
    if wheel:
        try:
            subprocess.check_call([sys.executable,'-m','pip','install','--quiet','--no-cache-dir',f'{wheel}==14.2.0'])
            importlib.invalidate_caches()
        except subprocess.CalledProcessError:
            print('CuPy installation failed. Auto backend will use NumPy when CUDA is unavailable.')
    else:
        print('CUDA version was not recognized. CPU execution remains available.')
for relative in ['XPOSim.py','xray_material_data/AtomicConstants.dat',
                 'xray_material_data/CrossSec-Compton_McMaster.dat','xray_material_data/f1f2_Windt.dat']:
    target = PROJECT_DIR / relative
    target.parent.mkdir(parents=True,exist_ok=True)
    if not target.exists():
        url = f'https://raw.githubusercontent.com/mamichberdey/XPOSim/{SOURCE_REF}/{relative}'
        with urllib.request.urlopen(url,timeout=90) as response:
            content = response.read()
        temporary = target.with_suffix(target.suffix+'.download')
        temporary.write_bytes(content)
        temporary.replace(target)
if hashlib.sha256((PROJECT_DIR/'XPOSim.py').read_bytes()).hexdigest()!=SOURCE_SHA256:
    raise RuntimeError('XPOSim checksum mismatch. Remove .xposim_runtime and repeat preparation.')
spec = importlib.util.spec_from_file_location('XPOSim',PROJECT_DIR/'XPOSim.py')
xpos = importlib.util.module_from_spec(spec)
sys.modules['XPOSim'] = xpos
spec.loader.exec_module(xpos)
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
print(f'XPOSim {xpos.__version__}; available backend: {xpos.select_backend("auto")}.')
print('Ready. Run cell 2 to open CRL and source inputs, then cell 3 for Results. Press Compute only after editing the inputs.')


In [ ]:
#@title 2 · CRL and source parameters — run this cell to show the input fields
import html
import math
import ipywidgets as widgets
from traitlets import TraitError
from IPython.display import display
try:
    from google.colab import output as colab_output
except ImportError:
    colab_output=None
if colab_output is not None:
    colab_output.enable_custom_widget_manager()

_previous_values={name:widget.value for name,widget in globals().get('controls',{}).items()}
_previous_values['accuracy']={'Quick':'Fast','Refined':'Accurate'}.get(_previous_values.get('accuracy'),_previous_values.get('accuracy','Fast'))
if 'material_mode' not in _previous_values:
    _previous_values['material_mode']='Chemical formula' if _previous_values.get('material')=='Custom' else 'Preset material'
if 'mesh_mode' not in _previous_values:
    _previous_values['mesh_mode']='Set point counts' if _previous_values.get('nx',0) or _previous_values.get('ny',0) else 'Auto'
if 'roi_mode' not in _previous_values:
    _previous_values['roi_mode']='Manual' if _previous_values.get('z_mode')=='Manual bounds' else 'Auto'
_first_explorer_initialization=not bool(globals().get('controls'))
_manual_density_value=globals().get('_manual_density_value',_previous_values.get('density',2.33) if _previous_values.get('density_mode')=='Manual' else 2.33)
_manual_density_user_set=globals().get('_manual_density_user_set',_previous_values.get('density_mode')=='Manual')
_preset_density_mode=globals().get('_preset_density_mode',_previous_values.get('density_mode','Auto'))
_coefficients_initialized=globals().get('_coefficients_initialized',False)
_show_detector=getattr(globals().get('detector_toggle'),'value',False)
_previous_editor=globals().get('parameter_editor')
if _previous_editor is not None:_previous_editor.close()
for widget in globals().get('controls',{}).values():
    widget.unobserve_all();widget.close()
controls={};containers={};field_labels={}

def input_field(name,label,widget,tooltip=''):
    if name in _previous_values:
        try:
            value=_previous_values[name]
            widget.value=value
        except (ValueError,TypeError,TraitError):pass
    controls[name]=widget
    widget.layout.width='100%'
    widget.layout.height='29px'
    caption=widgets.HTML('<span style="font-size:14px;font-weight:500;line-height:1.2" title="'+html.escape(tooltip,quote=True)+'">'+html.escape(label)+'</span>',layout=widgets.Layout(height='auto',min_height='27px',flex='1 0 auto',width='100%',overflow='visible',margin='0'))
    field_labels[name]=caption
    box=widgets.VBox([caption,widget],layout=widgets.Layout(width='100%',min_width='0',height='100%'))
    box.add_class('xposim-input-field')
    caption.add_class('xposim-field-caption')
    box.add_class('xposim-field-'+name)
    containers[name]=box
    return box

def group(fields,columns=4):
    return widgets.GridBox(fields,layout=widgets.Layout(
        grid_template_columns=f'repeat(auto-fit,minmax(min(100%,max({260 if columns==4 else 220}px,calc((100% - {14*(columns-1)}px) / {columns}))),1fr))',grid_gap='7px 12px',width='100%'))

def section(title,children,kind):
    parts=children
    if title:
        heading=widgets.HTML('<div style="font-size:16px;font-weight:650;letter-spacing:.01em;padding:1px 0 5px">'+html.escape(title)+'</div>')
        parts=[heading,*children]
    box=widgets.VBox(parts,layout=widgets.Layout(width='100%',min_width='0',padding='8px 12px',border='1px solid #d5dde5',margin='0'))
    box.add_class('xposim-section');box.add_class('xposim-section-'+kind)
    return box

source_controls=group([
    input_field('source','Source',widgets.Dropdown(options=['Point source','Gaussian source (incoherent)','Plane wave at CRL entrance'])),
    input_field('energy','Photon energy (keV)',widgets.FloatText(value=12.0)),
    input_field('distance','Source-to-CRL distance (m)',widgets.FloatText(value=15.0)),
    input_field('source_method','Extended-source method',widgets.Dropdown(options=['Gaussian convolution','Incoherent point sampling'])),
    input_field('fwhm_x','Source FWHM x (µm)',widgets.FloatText(value=5.0)),
    input_field('fwhm_y','Source FWHM y (µm)',widgets.FloatText(value=5.0)),
    input_field('source_points','Source nodes / axis',widgets.BoundedIntText(value=7,min=3,max=101,step=2)),
])
general_controls=group([
    input_field('focusing','CRL focusing',widgets.Dropdown(options=['One-dimensional focusing','Two-dimensional focusing'])),
    input_field('grid','Calculation grid',widgets.Dropdown(options=['1D','2D'])),
    input_field('accuracy','Sampling',widgets.Dropdown(options=['Fast','Accurate'],value='Fast')),
    input_field('elements','CRL elements',widgets.BoundedIntText(value=26,min=1,max=10000)),
])
geometry_controls=group([
    input_field('radius','Surface radius (µm)',widgets.FloatText(value=6.25)),
    input_field('aperture','Full aperture (µm)',widgets.FloatText(value=50.0)),
    input_field('bridge','Bridge thickness (µm)',widgets.FloatText(value=2.0)),
    input_field('depth','Structure depth y (µm)',widgets.FloatText(value=30.0)),
],2)
material_mode_field=input_field('material_mode','Material definition',widgets.Dropdown(options=['Preset material','Chemical formula','Direct δ / β']))
material_field=input_field('material','Preset material',widgets.Dropdown(options=['Beryllium (Be)','Silicon (Si)','Aluminium (Al)','Diamond (C)'],value='Silicon (Si)'))
formula_field=input_field('formula','Chemical formula',widgets.Text(value='SiO2',continuous_update=False))
density_field=input_field('density','Density (g/cm³)',widgets.FloatText(value=2.33))
density_mode=widgets.Dropdown(options=['Auto','Manual'],value=_previous_values.get('density_mode','Auto'),layout=widgets.Layout(width='94px',height='26px'))
density_mode.add_class('xposim-field-density_mode')
controls['density_mode']=density_mode
field_labels['density'].layout.width='auto'
density_field.children=(widgets.HBox([field_labels['density'],density_mode],layout=widgets.Layout(width='100%',min_height='27px',flex_flow='row wrap',justify_content='space-between',align_items='center')),controls['density'])
material_controls=group([material_mode_field,material_field,formula_field,density_field,
    input_field('delta','Manual δ',widgets.Text(value='3.4e-6',continuous_update=False),'Dimensionless refractive-index decrement; scientific notation is accepted.'),
    input_field('beta','Manual β',widgets.Text(value='3e-8',continuous_update=False),'Dimensionless absorption coefficient; beta may be zero.'),
],2)
advanced_fields=group([
    input_field('mesh_mode','Transverse grid control',widgets.Dropdown(options=['Auto','Set spacing','Set point counts'])),
    input_field('dx_nm','Target propagation Δx (nm)',widgets.FloatText(value=30.0),'Actual spacings are rounded to a scale-adaptive multiple of five and reported beside the backend.'),
    input_field('dy_nm','Target propagation Δy (nm)',widgets.FloatText(value=30.0)),
    input_field('z_spacing_mode','Axial grid control',widgets.Dropdown(options=['Auto','Set spacing'])),
    input_field('dz_mm','Target Δz (mm)',widgets.FloatText(value=.05),'Preferred axial steps use 1, 2.5 or 5 times a power of ten; the actual value is reported after calculation.'),
    input_field('adaptive_window','Adapt propagation window',widgets.Checkbox(value=True,indent=False)),
    input_field('nx','Grid points x (0 = auto)',widgets.IntText(value=0)),
    input_field('ny','Grid points y (0 = auto)',widgets.IntText(value=0)),
    input_field('window','Initial window / geometry',widgets.FloatText(value=1.3)),
    input_field('batch','Batch size (0 = auto)',widgets.IntText(value=0)),
    input_field('z_points','Target z sample count',widgets.BoundedIntText(value=51,min=5,max=501)),
    input_field('z_level','Axial ROI intensity fraction',widgets.BoundedFloatText(value=.2,min=.001,max=.99,step=.05)),
    input_field('xy_level','Transverse ROI intensity fraction',widgets.BoundedFloatText(value=.01,min=.00001,max=.5,step=.01)),
    input_field('tolerance','Peak z tolerance (µm; 0 = auto)',widgets.FloatText(value=0.0)),
])
roi_fields=group([
    input_field('roi_mode','Region selection',widgets.Dropdown(options=['Auto','Manual'])),
    input_field('z_min','Scan z minimum (mm)',widgets.FloatText(value=25.0)),
    input_field('z_max','Scan z maximum (mm)',widgets.FloatText(value=45.0)),
    input_field('x_min','x minimum (µm)',widgets.FloatText(value=-1.0)),
    input_field('x_max','x maximum (µm)',widgets.FloatText(value=1.0)),
    input_field('y_min','y minimum (µm)',widgets.FloatText(value=-1.0)),
    input_field('y_max','y maximum (µm)',widgets.FloatText(value=1.0)),
],4)
roi_status=widgets.HTML(layout=widgets.Layout(width='100%',min_height='30px'))
optical_layout=widgets.HTML(layout=widgets.Layout(width='100%'))
element_preview=widgets.HTML(layout=widgets.Layout(width='100%'))

PRESETS={'Beryllium (Be)':('Be',1.848),'Silicon (Si)':('Si',2.33),'Aluminium (Al)':('Al',2.70),'Diamond (C)':('C',3.515)}
_syncing_controls=False
_last_material_mode=controls['material_mode'].value
_preflight_result=None
preflight_geometry=widgets.HTML(layout=widgets.Layout(width='100%'))
preflight_material=widgets.HTML(layout=widgets.Layout(width='100%'))
preflight_status=widgets.HTML(layout=widgets.Layout(width='100%',min_height='23px',overflow='visible',margin='0'))
preflight_geometry.add_class('xposim-analytic-geometry')
preflight_material.add_class('xposim-analytic-material')
preflight_status.add_class('xposim-analytic-status')


def sync_controls(change=None):
    global _syncing_controls,_last_material_mode,_preset_density_mode,_coefficients_initialized
    if _syncing_controls:return
    _syncing_controls=True
    try:
        mode=controls['material_mode'].value
        preset=mode=='Preset material';formula=mode=='Chemical formula';direct=mode=='Direct δ / β'
        if mode!=_last_material_mode:
            if _last_material_mode=='Preset material':_preset_density_mode=controls['density_mode'].value
            if preset:controls['density_mode'].value=_preset_density_mode
            else:controls['density_mode'].value='Manual'
            if direct and not _coefficients_initialized and _preflight_result is not None:
                controls['delta'].value=f'{_preflight_result.delta:.10g}'
                controls['beta'].value=f'{_preflight_result.beta:.10g}'
                _coefficients_initialized=True
        _last_material_mode=mode
        if not preset:controls['density_mode'].value='Manual'
        if preset:_preset_density_mode=controls['density_mode'].value
        controls['material'].disabled=not preset
        controls['formula'].disabled=not formula
        controls['delta'].disabled=not direct;controls['beta'].disabled=not direct
        controls['density_mode'].disabled=not preset
        automatic=preset and controls['density_mode'].value=='Auto'
        controls['density'].disabled=direct or automatic
        if automatic:controls['density'].value=PRESETS[controls['material'].value][1]
        elif not direct:controls['density'].value=_manual_density_value
        label='Density (g/cm³)'
        field_labels['density'].value='<span style="font-size:14px;font-weight:500">'+label+'</span>'
        controls['density'].tooltip=('Automatic density for '+controls['material'].value) if automatic else ('Unused with direct δ / β' if direct else 'Manual material density')
        two_focusing=controls['focusing'].value=='Two-dimensional focusing'
        if two_focusing:controls['grid'].value='2D'
        controls['grid'].disabled=two_focusing
        two_grid=controls['grid'].value=='2D'
        gaussian=controls['source'].value=='Gaussian source (incoherent)'
        manual_roi=controls['roi_mode'].value=='Manual'
        active={'distance':controls['source'].value!='Plane wave at CRL entrance',
            'source_method':gaussian,'fwhm_x':gaussian,'fwhm_y':gaussian and two_grid,
            'source_points':gaussian and controls['source_method'].value=='Incoherent point sampling',
            'depth':two_grid and not two_focusing,'ny':two_grid and controls['mesh_mode'].value=='Set point counts',
            'nx':controls['mesh_mode'].value=='Set point counts',
            'dx_nm':controls['mesh_mode'].value=='Set spacing','dy_nm':two_grid and controls['mesh_mode'].value=='Set spacing',
            'dz_mm':controls['z_spacing_mode'].value=='Set spacing','z_points':controls['z_spacing_mode'].value=='Auto',
            'z_min':manual_roi,'z_max':manual_roi,'x_min':manual_roi,'x_max':manual_roi,
            'y_min':manual_roi and two_grid,'y_max':manual_roi and two_grid,
            'z_level':not manual_roi,'xy_level':not manual_roi}
        for name,enabled in active.items():controls[name].disabled=not enabled
    finally:_syncing_controls=False


def detector_bounds(v):
    if v['roi_mode']=='Auto':
        return None,None
    z=(v['z_min']*1e-3,v['z_max']*1e-3)
    x=(v['x_min']*1e-6,v['x_max']*1e-6)
    y=(v['y_min']*1e-6,v['y_max']*1e-6) if v['grid']=='2D' else None
    for axis,bounds in [('z',z),('x',x),*([('y',y)] if y is not None else [])]:
        if not all(map(math.isfinite,bounds)) or bounds[0]>=bounds[1]:
            raise ValueError(f'{axis} minimum must be finite and below its maximum.')
    if z[0]<0:
        raise ValueError('z minimum must be zero or positive.')
    return z,(x,y) if y is not None else x


def read_configuration(*,include_detector=True):
    v={name:widget.value for name,widget in controls.items()}
    scan_bounds,roi=detector_bounds(v) if include_detector else (None,None)
    names=['batch','tolerance']+(['nx']+(['ny'] if v['grid']=='2D' else []) if v['mesh_mode']=='Set point counts' else [])
    for name in names:
        if v[name]<0:raise ValueError(f'{name} must be non-negative; zero selects automatic control.')
    direct=v['material_mode']=='Direct δ / β'
    if direct:
        try:delta,beta=float(v['delta']),float(v['beta'])
        except ValueError:raise ValueError('Enter δ and β as numbers, for example 1.23e-6.') from None
        formula,density=None,None
    else:
        delta=beta=None
        formula=PRESETS[v['material']][0] if v['material_mode']=='Preset material' else v['formula']
        density=PRESETS[v['material']][1] if v['material_mode']=='Preset material' and v['density_mode']=='Auto' else v['density']
    gaussian=v['source']=='Gaussian source (incoherent)'
    return xpos.CRLFocusConfig(
        energy_kev=v['energy'],material_formula=formula,density_g_cm3=density,material_delta=delta,material_beta=beta,
        radius_m=v['radius']*1e-6,aperture_m=v['aperture']*1e-6,bridge_thickness_m=v['bridge']*1e-6,n_elements=v['elements'],
        geometry='two-dimensional' if v['focusing']=='Two-dimensional focusing' else 'one-dimensional',grid_dimension=2 if v['grid']=='2D' else 1,
        structure_depth_m=v['depth']*1e-6,source_distance_m=None if v['source']=='Plane wave at CRL entrance' else v['distance'],
        source_type={'Point source':'point','Gaussian source (incoherent)':'gaussian','Plane wave at CRL entrance':'plane'}[v['source']],
        source_fwhm_x_m=v['fwhm_x']*1e-6 if gaussian else 0.,source_fwhm_y_m=v['fwhm_y']*1e-6 if gaussian and v['grid']=='2D' else 0.,
        source_method='points' if v['source_method']=='Incoherent point sampling' else 'convolution',source_points=v['source_points'],backend='auto',
        accuracy='preview' if v['accuracy']=='Fast' else 'refined',
        grid_points=(v['nx'] or None) if v['mesh_mode']=='Set point counts' else None,
        grid_points_y=(v['ny'] or None) if v['grid']=='2D' and v['mesh_mode']=='Set point counts' else None,
        grid_step_x_m=v['dx_nm']*1e-9 if v['mesh_mode']=='Set spacing' else None,
        grid_step_y_m=v['dy_nm']*1e-9 if v['grid']=='2D' and v['mesh_mode']=='Set spacing' else None,
        scan_step_m=v['dz_mm']*1e-3 if v['z_spacing_mode']=='Set spacing' else None,
        window_factor=v['window'],adaptive_window=v['adaptive_window'],batch_size=v['batch'] or None,scan_points=v['z_points'],
        scan_range_m=scan_bounds,detector_roi_m=roi,
        axial_roi_level=v['z_level'],transverse_roi_level=v['xy_level'],peak_search_tolerance_m=v['tolerance']*1e-6 if v['tolerance'] else None)


def analytical_number(value,scientific=False):
    if value is None:return '—'
    if not math.isfinite(value):return '∞' if value>0 else '−∞'
    if scientific and value:
        mantissa,exponent=f'{value:.4e}'.split('e')
        return mantissa.rstrip('0').rstrip('.')+' × 10<sup>'+str(int(exponent))+'</sup>'
    return f'{value:.2f}'.rstrip('0').rstrip('.')


def analytical_length(value):
    if value is None:return '—'
    if not math.isfinite(value):return analytical_number(value)
    scale,unit=(1.,'m') if abs(value)>=.1 else ((1e-3,'mm') if abs(value)>=1e-4 else (1e-6,'µm'))
    return analytical_number(value/scale)+' '+unit


def analytical_entry(label,value):
    kind=' class="xposim-magnification"' if label=='Image-plane magnification' else ''
    return '<div'+kind+'><dt>'+html.escape(label)+'</dt><dd>'+value+'</dd></div>'


def magnification_row(axis,value):
    return (f'<div class="xposim-magnification-axis"><span class="xposim-axis-name">{axis}</span>'
            f'<span class="xposim-number">{analytical_number(value,True)}</span></div>')


def schematic_extent(value, reference, minimum, maximum):
    """Map a physical dimension monotonically into the illustrated range."""
    return minimum+(maximum-minimum)*value/(value+reference)


def schematic_material(cfg):
    return cfg.material_formula if cfg.material_formula is not None else 'Direct δ / β'


class Schematic:
    """Draw the same geometry in the notebook SVG and the printed report."""
    print_style={
        'xp-layout-axis':('#8293a6',None,.7,'--'),
        'xp-layout-dimension':('#516279',None,1,'-'),
        'xp-layout-lens':('#4d6788','#d9e5f3',1,'-'),
        'xp-layout-disc':('#597498','#edf2f8',.8,'-'),
        'xp-layout-slab':('#516279','#e7edf4',.9,'-'),
        'xp-layout-cavity':('#516279',None,.85,'-'),
        'xp-layout-hidden':('#9aa9ba',None,.7,'--'),
        'xp-layout-source-shape':('#a66798','#f9ebdd',.9,'-'),
        'xp-layout-ray':('#b7782e',None,1,'-'),
        'xp-layout-source':('#9d651f','#e5ae61',.8,'-'),
        'xp-layout-focus':('#12786e','#12786e',.9,'-'),
        'xp-layout-focus-line':('#12786e',None,1.5,'-'),
    }

    def __init__(self,width,height,ax=None):
        self.width,self.height=width,height
        self.ax=ax
        self.parts=[]
        if ax is not None:
            ax.set(xlim=(0,width),ylim=(height,0),aspect='equal')
            ax.set_axis_off()
            fig=ax.figure
            box=ax.get_position()
            self.print_scale=min(box.width*fig.get_figwidth()*72/width,box.height*fig.get_figheight()*72/height)

    def path(self,points,classes,*,close=False):
        self.compound((points,),classes,close=close)

    def compound(self,segments,classes,*,close=True):
        if self.ax is None:
            d=' '.join('M '+' L '.join(f'{x:.7g} {y:.7g}' for x,y in points)+(' Z' if close else '')
                       for points in segments)
            self.parts.append(f'<path class="{classes}" fill-rule="evenodd" d="{d}"/>')
            return
        from matplotlib.path import Path as PlotPath
        from matplotlib.patches import PathPatch
        stroke,fill,width,dash=self.print_style[classes.split()[0]]
        paths=[]
        orientation=None
        for points in segments:
            vertices=np.asarray(points)
            if close and len(segments)>1:
                x,y=vertices.T
                area=np.dot(x,np.roll(y,-1))-np.dot(y,np.roll(x,-1))
                if orientation is None:orientation=area
                elif area*orientation>0:vertices=vertices[::-1]
            paths.append(PlotPath(np.vstack((vertices,vertices[0])),closed=True) if close else PlotPath(vertices))
        self.ax.add_patch(PathPatch(PlotPath.make_compound_path(*paths),facecolor=fill or 'none',
            edgecolor=stroke,linewidth=width*.72,linestyle=dash,zorder=2))

    def label(self,x,y,text,classes='xp-layout-label',anchor='middle'):
        if self.ax is None:
            self.parts.append(f'<text class="{classes}" x="{x:.4g}" y="{y:.4g}" text-anchor="{anchor}">{html.escape(text)}</text>')
            return
        small=classes.startswith('xp-layout-small')
        size=max(5.8,min(10,(14 if small else 17)*self.print_scale))
        self.ax.text(x,y,text,ha={'start':'left','middle':'center','end':'right'}[anchor],
            va='baseline',fontsize=size,color='#41546b' if small else '#172235',
            fontweight='normal' if small else 'semibold',zorder=4,usetex=False,parse_math=False)

    def dot(self,x,y,radius,classes):
        if self.ax is None:
            self.parts.append(f'<circle class="{classes}" cx="{x:.5g}" cy="{y:.5g}" r="{radius}"/>')
            return
        from matplotlib.patches import Circle
        stroke,fill,lw,_=self.print_style[classes]
        self.ax.add_patch(Circle((x,y),radius,facecolor=fill,edgecolor=stroke,linewidth=lw*.72,zorder=3))

    def ruler(self,start,stop,offset,classes=''):
        """Offset a dimension line and draw the two measured endpoints."""
        a,b,shift=np.asarray(start),np.asarray(stop),np.asarray(offset)
        delta=b-a
        perpendicular=np.array([-delta[1],delta[0]])/np.linalg.norm(delta)
        style='xp-layout-dimension '+classes
        p,q=a+shift,b+shift
        extension=shift*(4/max(4,np.linalg.norm(shift)))
        self.path([a,p+extension],style)
        self.path([b,q+extension],style)
        self.path([p,q],style)
        for end in (p,q):
            self.path([end-3*perpendicular,end+3*perpendicular],style)
        return (p+q)/2

    def html(self,css,description,defs=''):
        return (f'<div class="{css}"><svg role="img" aria-label="{description}" viewBox="0 0 {self.width} {self.height}">'
                +defs+''.join(self.parts)+'</svg></div>')


def element_schematic(cfg,result,*,ax=None,show_material=True):
    """The z–x profile uses one scale. Depth y is projected at 45 degrees."""
    drawing=Schematic(418,242,ax)
    depth=cfg.structure_depth_m if cfg.grid_dimension==2 and cfg.geometry=='one-dimensional' else None
    aperture,length,bridge=cfg.aperture_m,result.element_length_m,cfg.bridge_thickness_m
    offset=(depth or 0)/math.sqrt(2)
    scale=min(218/(length+offset),132/(aperture+offset))
    depth_projection=offset*scale
    center_x=153-depth_projection/2
    center_y=127+depth_projection/2
    left,right=center_x-length*scale/2,center_x+length*scale/2
    top,bottom=center_y-aperture*scale/2,center_y+aperture*scale/2
    transverse=np.linspace(-aperture/2,aperture/2,65)
    longitudinal=(bridge/2+transverse**2/(2*cfg.radius_m))*scale
    flank=np.column_stack((center_x-longitudinal,center_y+transverse*scale))
    outline=np.vstack((flank,np.column_stack((center_x+longitudinal,center_y+transverse*scale))[::-1]))
    if depth is not None:
        extrude=np.array((depth_projection,-depth_projection))
        drawing.path(outline+extrude,'xp-layout-hidden xp-element-depth',close=True)
        for corner in (outline[0],outline[64],outline[65],outline[-1]):
            drawing.path([corner,corner+extrude],'xp-layout-hidden')
    drawing.path(outline,'xp-layout-lens xp-element-outline',close=True)
    drawing.path([(left-7,center_y),(right+depth_projection+7,center_y)],'xp-layout-axis')

    def dimension(symbol,start,stop,offset,style,label_offset):
        if np.linalg.norm(np.subtract(stop,start))<21:
            return
        x,y=drawing.ruler(start,stop,offset,style)
        drawing.label(x+label_offset[0],y+label_offset[1],symbol,'xp-layout-small xp-dimension-symbol')

    dimension('D',(left,top),(left,bottom),(-14,0),'xp-element-aperture',(-10,4))
    dimension('p',(left,bottom),(right,bottom),(0,17),'xp-element-length',(0,15))
    t=bridge*scale/2
    dimension('t',(center_x-t,center_y),(center_x+t,center_y),(0,-13),'xp-element-bridge',(0,-6))
    radius=cfg.radius_m*scale
    origin=center_x-t-radius
    if origin>=18:
        dimension('R',(origin,center_y),(center_x-t,center_y),(0,14),'xp-element-radius',(0,15))
    if depth is not None:
        dimension('y',(right,bottom),(right+depth_projection,bottom-depth_projection),(9,9),
                  'xp-element-depth-ruler',(11,9))

    heading='Circular element section' if cfg.geometry=='two-dimensional' else 'Planar element section'
    drawing.label(209,20,heading)
    if show_material:drawing.label(209,39,schematic_material(cfg),'xp-layout-small')
    values=[('Aperture D',aperture),('Vertex radius R',cfg.radius_m),('Bridge t',bridge),('Element length p',length)]
    if depth is not None:values.append(('Depth y',depth))
    for i,(label,value) in enumerate(values):
        y=61+35*i
        drawing.label(283,y,label,'xp-layout-small','start')
        drawing.label(283,y+15,f'{value*1e6:.5g} µm','xp-layout-small','start')
    return drawing.html('xp-element-view','CRL element section to scale')


def optical_schematic(cfg,result,*,ax=None,show_depth=True):
    """Draw the optical train in projection or isometry."""
    iso=cfg.grid_dimension==2
    round_lens=cfg.geometry=='two-dimensional'
    real_image=result.real_image
    plane=cfg.source_type=='plane'
    extended=cfg.source_type=='gaussian'
    source_label={'point':'Point source','plane':'Plane wave','gaussian':'Gaussian source'}[cfg.source_type]
    lens_label='Two-axis CRL' if round_lens else 'One-axis CRL'
    source_z=69 if plane else 37+64/(1+cfg.source_distance_m/15)
    count=min(cfg.n_elements,4)
    half_length=schematic_extent(result.lens_length_m,.00265,31,115)
    entrance,exit_z=287-half_length,287+half_length
    pitch=(exit_z-entrance)/count
    distance=result.image_distance_m if real_image else .034
    focus=exit_z+(553-exit_z)*schematic_extent(distance,.034,.54,.94)
    half_aperture=schematic_extent(cfg.aperture_m,50e-6,15,38)
    slab_height=half_aperture+7
    depth=schematic_extent(cfg.structure_depth_m,30e-6,7,22) if iso and not round_lens else 0
    web_fraction=min(.6,max(.12,cfg.bridge_thickness_m/result.element_length_m))
    drawing=Schematic(600,215,ax)

    def project(z,x=0,y=0):
        if iso:
            # Three projected axes meet at 120°, with z horizontal.
            return z-.5*(x+y),125+math.sqrt(3)/2*(y-x)
        return z,125-x

    def path(points,style,close=False):
        drawing.path([project(*p) for p in points],style,close=close)

    def line(a,b,style):
        path([a,b],style)

    label=drawing.label

    def dot(p,r,style):
        drawing.dot(*project(*p),r,style)

    levels=[half_aperture*(i/16-1) for i in range(33)]

    def plate_face(y):
        return [(entrance,-slab_height,y),(exit_z,-slab_height,y),
                (exit_z,slab_height,y),(entrance,slab_height,y)]

    def cavity(z,y):
        # Adjacent parabolic surfaces bound the void between two material webs.
        radius_z=pitch*(1-web_fraction)/2
        left=[(z-radius_z*(1-(x/half_aperture)**2),x,y) for x in levels]
        right=[(z+radius_z*(1-(x/half_aperture)**2),x,y) for x in reversed(levels)]
        return left+right

    def draw_planar_stack():
        stations=[entrance+(i+.5)*pitch for i in range(count)]
        front=plate_face(depth)
        if iso:
            rear=plate_face(-depth)
            path(rear,'xp-layout-hidden',True)
            path([rear[3],rear[2],front[2],front[3]],'xp-layout-slab',True)
            path([rear[0],rear[3],front[3],front[0]],'xp-layout-slab',True)
        holes=[cavity(z,depth) for z in stations]
        faces=[[project(*p) for p in surface] for surface in (front,*holes)]
        drawing.compound(faces,'xp-layout-slab')
        if ax is None:
            drawing.compound(faces,'xp-crl-hatch')
        for z,hole in zip(stations,holes):
            if iso:
                path(cavity(z,depth/4),'xp-layout-hidden',True)
                path(cavity(z,-depth/2),'xp-layout-hidden',True)
            path(hole,'xp-layout-cavity xp-layout-opening',True)

    def draw_round_stack():
        # At each radius, the two circular surface contours meet a thicker web.
        angles=[i*math.tau/40 for i in range(41)]
        for i in range(count):
            center=entrance+(i+.5)*pitch
            half_cell=.46*pitch
            half_web=half_cell*web_fraction
            for sign in (1,-1):
                for radial_fraction in (1,.76,.5):
                    r=half_aperture*radial_fraction
                    z=center+sign*(half_web+(half_cell-half_web)*radial_fraction**2)
                    contour=[(z,r*math.cos(a),r*math.sin(a)) for a in angles]
                    path(contour,'xp-layout-disc' if radial_fraction==1 else 'xp-layout-cavity',True)
            extent=half_aperture/math.sqrt(2)
            for x,y in ((extent,-extent),(-extent,extent)):
                line((center-half_cell,x,y),(center+half_cell,x,y),'xp-layout-cavity')

    line((28,0,0),(557,0,0),'xp-layout-axis')
    label(571,129,'z','xp-layout-small')

    if extended:
        if iso:
            size_x=schematic_extent(cfg.source_fwhm_x_m,5e-6,5,35)
            size_y=schematic_extent(cfg.source_fwhm_y_m,5e-6,4,27)
            source_face=[(source_z,size_x*math.cos(t*math.tau/48),size_y*math.sin(t*math.tau/48)) for t in range(49)]
            path(source_face,'xp-layout-source-shape',True)
        else:
            sigma=schematic_extent(cfg.source_fwhm_x_m,5e-6,3,15)
            xs=[sigma*i/6 for i in range(-18,19)]
            source_face=[(source_z+16*math.exp(-.5*(x/sigma)**2),x,0) for x in xs]
            path([(source_z,xs[0],0),*source_face,(source_z,xs[-1],0)],'xp-layout-source-shape',True)
        if cfg.source_method=='points':
            width= size_x if iso else 2*sigma
            for offset in (-.85,-.42,0,.42,.85):
                dot((source_z,width*offset,0),1.8,'xp-layout-source')
    elif not plane:
        dot((source_z,0,0),5,'xp-layout-source')

    if round_lens:
        draw_round_stack()
    else:
        draw_planar_stack()

    ray_x=.76*half_aperture
    ray_y=.72*(half_aperture if round_lens else depth)
    ray_points=[(x,y) for x in (-ray_x,ray_x) for y in ((-ray_y,ray_y) if iso else (0,))]
    for x,y in ray_points:
        start=(30,x,y) if plane else (source_z,x*.55 if extended else 0,y*.5 if extended else 0)
        incident_edge=(entrance,x,y)
        exit_y=y*.88 if round_lens else y
        exit_edge=(exit_z,x*.82,exit_y)
        if real_image:
            final_y=0 if round_lens else y
            path([start,incident_edge,exit_edge,(focus,0,final_y),(551,-x*.20,final_y)],'xp-layout-ray')
        else:
            path([start,incident_edge,exit_edge,(535,x*1.32,exit_y)],'xp-layout-ray')

    label(12,30,source_label,anchor='start')
    lens_label_z=(entrance+exit_z)/2+20
    label(lens_label_z,30,lens_label)
    label(lens_label_z,47,f'{cfg.n_elements} elements','xp-layout-small')
    label(lens_label_z,63,schematic_material(cfg),'xp-layout-small')
    if real_image:
        focus_label='Focal line' if iso and not round_lens else 'Image point'
        if iso and not round_lens:
            line((focus,0,-23),(focus,0,23),'xp-layout-focus-line')
        else:
            dot((focus,0,0),5,'xp-layout-focus')
        label(min(focus,518),30,focus_label)
        drawing.ruler((exit_z,184),(focus,184),(0,6))
        label((exit_z+focus)/2,207,'Image '+html.unescape(analytical_length(result.image_distance_m)),'xp-layout-small')
    else:
        image_label=('Image at infinity' if math.isinf(result.image_distance_m or 0) else 'Virtual image')
        label(587,30,image_label if result.external_focus else 'Focus inside CRL',anchor='end')
    if not plane:
        drawing.ruler((source_z,184),(entrance,184),(0,6))
        label((source_z+entrance)/2,207,analytical_number(cfg.source_distance_m)+' m to CRL entrance','xp-layout-small')
    if extended:
        widths='x '+analytical_number(cfg.source_fwhm_x_m*1e6)
        if iso:
            widths+=' · y '+analytical_number(cfg.source_fwhm_y_m*1e6)
        label(12,51,'FWHM '+widths+' µm','xp-layout-small','start')
    if iso and not round_lens and show_depth:
        near,far=[project(entrance,-slab_height,y) for y in (-depth,depth)]
        _,y=drawing.ruler(near,far,(-25,10),'xp-layout-depth-ruler')
        label(min(near[0],far[0])-34,y+5,'Depth y · '+analytical_number(cfg.structure_depth_m*1e6)+' µm',
              'xp-layout-small','end')
    line((546,-36,0),(546,-19,0),'xp-layout-axis')
    label(*project(546,-15,0),'x','xp-layout-small')
    if iso:
        line((546,-36,0),(546,-36,22),'xp-layout-axis')
        px,py=project(546,-36,29)
        label(px,py+4,'y','xp-layout-small')

    # Vector fills for the source and the plate material.
    definitions=(
        '<defs><pattern id="xp-crl-hatch" width="5" height="5" patternUnits="userSpaceOnUse">'
        '<path class="xp-layout-hatch" d="M -1 1 L 1 -1 M 0 5 L 5 0 M 4 6 L 6 4"/></pattern>'
        '<radialGradient id="xp-gaussian" cx=".5" cy=".5" r=".5">'
        '<stop offset="0" stop-color="#ffdf7f"/><stop offset=".5" stop-color="#e1949f" stop-opacity=".8"/>'
        '<stop offset="1" stop-color="#a367c6" stop-opacity=".15"/></radialGradient></defs>')
    description='Optical layout with '+source_label+' and '+lens_label+(' in isometric view' if iso else ' in x-z projection')
    return drawing.html('xp-optical-layout',description,definitions)


def update_roi_status():
    values={name:widget.value for name,widget in controls.items()}
    color='var(--xp-secondary)'
    try:
        z,_=detector_bounds(values)
        note=('z is measured from the CRL exit. x and y are measured from the optical axis.' if z is not None else
              'The beam determines the axial interval and transverse region automatically.')
    except ValueError as exc:
        note=str(exc);color='#b77d38'
    roi_status.value='<div class="xp-layout-caption" style="color:'+color+'">'+html.escape(note)+'</div>'


def update_analytical_preview():
    global _preflight_result
    try:
        cfg=read_configuration(include_detector=False);result=xpos.analyze_crl(cfg);_preflight_result=result
        optical_layout.value=optical_schematic(cfg,result)
        element_preview.value=element_schematic(cfg,result)
        web=analytical_number(100*result.web_transmission)+' %'
        rocking='Unresolved' if result.rocking_fwhm_rad is None else analytical_number(1e3*result.rocking_fwhm_rad)+' mrad'
        focal=analytical_length(result.collimated_focus_m);image=analytical_length(result.image_distance_m)
        maximum=str(result.max_element_count);length=analytical_length(result.lens_length_m)
        delta=analytical_number(result.delta,True);beta=analytical_number(result.beta,True)
        scales=result.magnification_at_image
        if not any(value is not None for value in scales):
            magnification='—'
        elif cfg.geometry=='two-dimensional' and all(value is not None for value in scales) and math.isclose(*scales,rel_tol=1e-9):
            magnification=magnification_row('x, y',scales[0])
        else:
            labels=() if cfg.grid_dimension==1 else (
                'x','Source scale y' if cfg.geometry=='one-dimensional' else 'y')
            magnification=''.join(magnification_row(labels[i] if labels else '',value)
                                  for i,value in enumerate(scales))
        if not result.external_focus:message=f'External focus requires N ≤ {result.max_element_count}; adjust CRL elements.'
        elif not result.real_image:message='This source/lens combination has a virtual or infinite image distance.'
        else:message='Analytical distances are measured from the CRL exit.'
        color='var(--xp-secondary)' if result.real_image else '#b77d38'
    except Exception as exc:
        _preflight_result=None
        optical_layout.value='<div class="xp-layout-caption">Enter valid source and CRL parameters to preview the layout.</div>'
        element_preview.value=''
        focal=image=maximum=length=delta=beta=magnification=web=rocking='—';message=str(exc);color='#b77d38'
    grid='<dl class="xposim-analytical-list">'
    preflight_geometry.value=grid+''.join([analytical_entry('Analytical focal distance',focal),analytical_entry('Analytical image distance',image),analytical_entry('Image-plane magnification',magnification),analytical_entry('Maximum CRL elements',maximum),analytical_entry('Full CRL length',length)])+'</dl>'
    preflight_material.value=(grid+analytical_entry('δ · refractive decrement',delta)+analytical_entry('β · absorption index',beta)+
        analytical_entry('Web transmission',web)+analytical_entry('Rocking curve FWHM (analytical)',rocking)+'</dl>')
    preflight_status.value='<div style="font-size:12px;line-height:1.3;color:'+color+';padding-top:4px">'+html.escape(message)+'</div>'


def inputs_changed(change=None):
    global _manual_density_value,_manual_density_user_set,_coefficients_initialized
    if _syncing_controls:return
    if change and change['owner'] is controls['density_mode'] and change['new']=='Manual' and controls['material_mode'].value=='Preset material' and not _manual_density_user_set:
        _manual_density_value=controls['density'].value
    if change and change['owner'] is controls['density'] and not controls['density'].disabled:
        _manual_density_value=change['new'];_manual_density_user_set=True
    if change and change['owner'] in (controls['delta'],controls['beta']) and not controls['delta'].disabled:
        _coefficients_initialized=True
    sync_controls(change)
    update_analytical_preview()
    update_roi_status()
    if 'sync_results_notice' in globals():sync_results_notice()
for widget in controls.values():widget.observe(inputs_changed,names='value')
sync_controls();update_analytical_preview();update_roi_status()

layout_section=section('Optical layout',[optical_layout],'optical-layout')
analytical_section=section('Analytical estimates',[preflight_geometry,preflight_status],'analytical')
overview=widgets.GridBox([layout_section,analytical_section],layout=widgets.Layout(width='100%'))
overview.add_class('xp-parameter-overview')
source_section=section('Source',[source_controls],'source')
geometry_inputs=widgets.VBox([general_controls,geometry_controls],layout=widgets.Layout(width='100%',gap='7px'))
geometry_layout=widgets.GridBox([geometry_inputs,element_preview],layout=widgets.Layout(width='100%'))
geometry_layout.add_class('xp-geometry-fields')
geometry_section=section('CRL geometry',[geometry_layout],'geometry')

material_layout=widgets.GridBox([material_controls,preflight_material],layout=widgets.Layout(width='100%'))
material_layout.add_class('xp-material-fields')
material_section=section('Material',[material_layout],'material')

roi_section=section('Calculation region',[roi_fields,roi_status,
    widgets.HTML('<b>Sampling and peak search</b>'),advanced_fields],'detector-region')
detector_toggle=widgets.Checkbox(value=_show_detector,description='Show scan region and sampling settings',indent=False,
    layout=widgets.Layout(width='100%',height='auto'))
detector_toggle.add_class('xp-detector-toggle')

def show_detector_settings(change):
    roi_section.layout.display='' if change['new'] else 'none'

detector_toggle.observe(show_detector_settings,names='value')
show_detector_settings({'new':detector_toggle.value})
for panel in (source_controls,general_controls,geometry_controls,material_controls,roi_fields,advanced_fields):
    panel.add_class('xp-control-grid')
source_controls.add_class('xp-source-grid')

UI_CSS='<style>\n.xposim-parameters,.xposim-explorer {\n --xp-bg:var(--colab-primary-surface-color,light-dark(#f3f6fa,#181d25));\n --xp-surface:var(--colab-secondary-surface-color,light-dark(#ffffff,#222a35));\n --xp-fg:var(--colab-primary-text-color,light-dark(#172235,#f1f5fa));\n --xp-secondary:var(--colab-secondary-text-color,light-dark(#465b74,#c0ccdd));\n --xp-border:light-dark(#9aaabe,#64768f);--xp-grid:light-dark(#d6dee9,#3c495c);\n --xp-heading:light-dark(#e6edf5,#303c4f);--xp-input:light-dark(#fff,#293445);\n box-sizing:border-box;color:var(--xp-fg);background:var(--xp-bg);color-scheme:light dark;padding:10px;\n}\n.xposim-parameters.xposim-dark,.xposim-explorer.xposim-dark {\n --xp-bg:#181d25;--xp-surface:#222a35;--xp-fg:#f1f5fa;--xp-secondary:#c0ccdd;\n --xp-border:#64768f;--xp-grid:#3c495c;--xp-heading:#303c4f;--xp-input:#293445;color-scheme:dark;\n}\n.xposim-parameters.xposim-light,.xposim-explorer.xposim-light {\n --xp-bg:#f3f6fa;--xp-surface:#fff;--xp-fg:#172235;--xp-secondary:#465b74;\n --xp-border:#9aaabe;--xp-grid:#d6dee9;--xp-heading:#e6edf5;--xp-input:#fff;color-scheme:light;\n}\n.xposim-parameters .widget-box,.xposim-explorer .widget-box,\n.xposim-parameters .widget-inline-hbox,.xposim-explorer .widget-inline-hbox,\n.xposim-parameters .widget-html,.xposim-explorer .widget-html {\n box-sizing:border-box;min-width:0;max-width:100%;margin:0;overflow:visible;\n}\n.xposim-parameters .widget-html,.xposim-explorer .widget-html {color:var(--xp-fg)}\n.xposim-parameters .widget-html-content *,.xposim-explorer .widget-html-content * {color:inherit}\n.xposim-parameters .widget-html-content,.xposim-explorer .widget-html-content {\n min-width:0;max-width:100%;overflow-wrap:normal;word-break:normal;color:inherit;\n}\n.xposim-parameters input,.xposim-parameters select,.xposim-explorer input,.xposim-explorer select {\n min-width:0;box-sizing:border-box;background:var(--xp-input);color:var(--xp-fg);border-color:var(--xp-border);\n}\n.xposim-parameters input:disabled,.xposim-parameters select:disabled {opacity:.58}\n.xposim-parameters .widget-label,.xposim-explorer .widget-label {color:var(--xp-fg)}\n.xposim-explorer .widget-image {display:block;min-width:0;max-width:100%;margin:0;height:auto}\n.xposim-parameters .xposim-section {border:1px solid var(--xp-border)!important;border-radius:4px;background:var(--xp-surface)}\n.xposim-analytical-list {margin:10px 0 0;padding:0;font-size:14px;line-height:1.4}\n.xposim-analytical-list>div {display:grid;grid-template-columns:minmax(0,1fr) auto;align-items:baseline;gap:6px 12px;padding:4px 0;border-bottom:1px solid var(--xp-grid)}\n.xposim-analytical-list>div:last-child {border-bottom:0}\n.xposim-parameters .xposim-analytical-list dt,.xposim-parameters .xposim-analytical-list dd {width:auto;float:none;clear:none;min-width:0;padding:0;margin:0}\n.xposim-parameters .xposim-analytical-list dt {font-weight:400;color:var(--xp-secondary)}\n.xposim-parameters .xposim-analytical-list dd {font-size:19px;font-weight:600;white-space:nowrap;font-variant-numeric:tabular-nums}\n.xposim-analytic-status,.xposim-material-note {color:var(--xp-secondary)}\n.xposim-explorer .xposim-run {border:1px solid var(--xp-border);margin:7px 0 8px;background:var(--xp-surface)}\n.xposim-explorer .xposim-group-title {font-size:14px;font-weight:600;padding:5px 10px;background:var(--xp-heading);border-bottom:1px solid var(--xp-border)}\n.xposim-explorer .xposim-run-table {width:100%;border-collapse:collapse;table-layout:fixed;font-variant-numeric:tabular-nums}\n.xposim-explorer .xposim-run-table th {font-size:14px;font-weight:500;text-align:left;vertical-align:bottom;padding:6px 10px;color:var(--xp-fg)}\n.xposim-explorer .xposim-run-table td {font-size:21px;font-weight:600;text-align:left;padding:1px 10px 7px}\n.xposim-explorer .xposim-run-table th+th,.xposim-explorer .xposim-run-table td+td {border-left:1px solid var(--xp-border)}\n.xposim-explorer .xposim-unit {display:block;font-size:12px;font-weight:400;color:var(--xp-secondary)}\n.xposim-explorer .xposim-panel {border:1px solid var(--xp-border);background:var(--xp-surface);min-width:0}\n.xposim-explorer .xposim-panel-heading {display:flex;gap:10px;align-items:baseline;justify-content:space-between;flex-wrap:wrap;font-size:15px;font-weight:600;padding:7px 10px;background:var(--xp-heading);border-bottom:1px solid var(--xp-border)}\n.xposim-explorer .xposim-selected-z {font-size:20px;font-weight:600;font-variant-numeric:tabular-nums}\n.xposim-explorer .xposim-plane-metrics {display:grid;grid-template-columns:minmax(0,.9fr) minmax(0,1.1fr);margin:0 9px 8px;gap:9px}\n.xposim-explorer .xposim-metric-box {border:1px solid var(--xp-border);background:var(--xp-surface);min-width:0}\n.xposim-explorer .xposim-metric-title {font-size:14px;font-weight:500;padding:4px 8px;background:var(--xp-heading);border-bottom:1px solid var(--xp-border)}\n.xposim-explorer .xposim-metric-value {font-size:21px;font-weight:600;padding:4px 8px;font-variant-numeric:tabular-nums}\n.xposim-explorer .xposim-axis-table {width:100%;border-collapse:collapse;table-layout:fixed;font-size:14px;font-variant-numeric:tabular-nums}\n.xposim-explorer .xposim-axis-table th {font-size:12px;font-weight:400;text-align:right;padding:2px 8px;color:var(--xp-secondary)}\n.xposim-explorer .xposim-axis-table td {font-size:19px;font-weight:600;text-align:right;padding:2px 8px}\n.xposim-explorer .xposim-axis-table th:first-child,.xposim-explorer .xposim-axis-table td:first-child {text-align:left;width:43px}\n.xposim-explorer .xposim-axis-table td:first-child {font-size:13px;font-weight:400}\n.xposim-explorer .xposim-axis-table tr+tr {border-top:1px solid var(--xp-grid)}\n.xposim-axis-badge {display:inline-flex;align-items:center;gap:5px;white-space:nowrap}\n.xposim-axis-badge>span {display:inline-block;width:12px}\n.xposim-explorer .xposim-selection-controls {padding:6px 8px;margin:8px 0 5px;border:1px solid var(--xp-border);background:var(--xp-heading)}\n.xposim-explorer .jupyter-button {border:1px solid var(--xp-border);border-radius:4px;background:var(--xp-input);color:var(--xp-fg)}\n.xposim-explorer .jupyter-button:focus-visible {outline:2px solid #38c5b6;outline-offset:2px}\n.xposim-explorer .widget-button.mod-primary {background:#355f85;color:#fff}\n.xposim-explorer .noUi-target {background:var(--xp-border)}\n.xposim-explorer .xposim-notice {font-size:13px;padding:5px 8px;border-left:3px solid var(--xp-border);background:var(--xp-heading)}\n.xposim-explorer .xposim-runtime {color:var(--xp-secondary)}\n.xposim-explorer .xposim-runtime b {color:var(--xp-fg)}\n.xposim-parameters .widget-html-content,.xposim-parameters .widget-html-content *,\n.xposim-explorer .widget-html-content,.xposim-explorer .widget-html-content * {line-height:1.3}\n.xposim-explorer .xposim-run {margin:4px 0 5px}\n.xposim-explorer .xposim-group-title {padding:3px 10px;font-size:13px}\n.xposim-explorer .xposim-run-table th .xposim-unit {display:inline;margin-left:5px}\n.xposim-explorer .xposim-metric-title .xposim-unit {display:inline;margin-left:5px}\n.xposim-explorer .xposim-axis-table td {padding-top:0;padding-bottom:0}\n\n.xposim-parameters,.xposim-explorer,\n.xposim-parameters .widget-html-content,.xposim-explorer .widget-html-content,\n.xposim-parameters .widget-label,.xposim-explorer .widget-label,\n.xposim-parameters input,.xposim-parameters select,.xposim-explorer input,.xposim-explorer select,\n.xposim-explorer .jupyter-button {\n font-family:Consolas,\'DejaVu Sans Mono\',\'Liberation Mono\',monospace;\n}\n.xposim-parameters .widget-html-content *,.xposim-explorer .widget-html-content * {font-family:inherit}\n.xposim-explorer .xposim-single-metric {grid-template-columns:1fr}\n\n/* The output width can be smaller than the viewport (Colab side panels/iframes). */\n.xposim-parameters {container:xp-parameters / inline-size}\n.xposim-explorer {container:xp-explorer / inline-size}\n.xposim-parameters .xposim-section {container:xp-section / inline-size}\n.xposim-parameters .xposim-field-caption {height:auto;min-height:27px}\n.xposim-parameters .xposim-input-field>input,\n.xposim-parameters .xposim-input-field>select {flex-shrink:0}\n.xposim-parameters .xposim-analytical-list dd {white-space:normal}\n.xposim-parameters .xposim-number {white-space:nowrap;display:inline-block}\n.xposim-parameters .xposim-analytical-list .xposim-magnification {grid-template-columns:1fr}\n.xposim-magnification-axis {display:flex;flex-wrap:wrap;align-items:baseline;gap:2px 12px}\n.xposim-magnification-axis+.xposim-magnification-axis {margin-top:4px}\n.xposim-magnification-axis .xposim-axis-name {font-size:13px;font-weight:400;white-space:nowrap}\n.xposim-explorer .xposim-run-metrics {display:grid;grid-template-columns:repeat(var(--xp-metric-count),minmax(0,1fr));margin:0;padding:0}\n.xposim-explorer .xposim-run-metrics>div {display:flex;flex-direction:column;min-width:0;padding:7px 10px;border-right:1px solid var(--xp-border)}\n.xposim-explorer .xposim-run-metrics>div:last-child {border-right:0}\n.xposim-explorer .xposim-run-metrics dt {flex:1;float:none;width:auto;margin:0;padding:0 0 5px;font-size:14px;line-height:1.4;font-weight:500}\n.xposim-explorer .xposim-run-metrics dd {float:none;width:auto;margin:0;padding:0;font-size:21px;line-height:1.3;font-weight:600;white-space:nowrap;font-variant-numeric:tabular-nums}\n.xposim-explorer .xposim-run-metrics .xposim-unit {display:inline;margin-left:6px;white-space:nowrap}\n.xposim-explorer .xposim-toolbar {display:grid;grid-template-columns:125px minmax(0,1fr) 160px;grid-template-areas:"compute runtime theme"}\n.xposim-toolbar>.widget-button {grid-area:compute}\n.xposim-toolbar>.xposim-runtime {grid-area:runtime}\n.xposim-toolbar>.xposim-theme-selector {grid-area:theme}\n.xposim-explorer .xposim-selected-z,.xposim-explorer .xposim-plane-label {white-space:nowrap}\n.xposim-explorer .xposim-selection-row {gap:4px}\n@container xp-section (max-width:460px) {\n .xposim-analytical-list>div {grid-template-columns:1fr;gap:3px}\n .xposim-parameters .xposim-analytical-list dd {font-size:17px}\n}\n@container xp-parameters (max-width:400px) {\n .xposim-parameters .xposim-section {padding:10px 8px!important}\n}\n@container xp-explorer (max-width:900px) {\n .xposim-explorer .xposim-run-metrics {grid-template-columns:repeat(2,minmax(0,1fr))}\n .xposim-explorer .xposim-run-metrics>div {border-right:0;border-bottom:1px solid var(--xp-border)}\n .xposim-explorer .xposim-run-metrics>div:nth-child(odd) {border-right:1px solid var(--xp-border)}\n .xposim-explorer .xposim-run-metrics>div:last-child:nth-child(odd) {grid-column:1/-1;border-right:0}\n}\n@container xp-explorer (max-width:700px) {\n .xposim-explorer .xposim-toolbar {grid-template-columns:minmax(90px,1fr) minmax(100px,1fr);grid-template-areas:"compute theme" "runtime runtime";gap:8px!important}\n .xposim-explorer .xposim-theme-selector {justify-self:end}\n .xposim-explorer .xposim-runtime {width:100%!important}\n}\n@container xp-explorer (max-width:540px) {\n .xposim-explorer .xposim-run-metrics {grid-template-columns:1fr}\n .xposim-explorer .xposim-run-metrics>div:nth-child(odd) {border-right:0}\n .xposim-explorer .xposim-run-metrics>div:last-child {border-bottom:0}\n .xposim-explorer .xposim-selection-row {display:grid!important;grid-template-columns:34px minmax(0,1fr) minmax(0,1fr) 34px;gap:8px 4px}\n .xposim-explorer .xposim-previous-plane {grid-column:1;grid-row:1}\n .xposim-explorer .xposim-plane-slider {grid-column:2/4;grid-row:1;width:100%!important;min-width:0!important}\n .xposim-explorer .xposim-next-plane {grid-column:4;grid-row:1}\n .xposim-explorer .xposim-peak-plane {grid-column:1/3;grid-row:2}\n .xposim-explorer .xposim-z-input {grid-column:3/5;grid-row:2;width:100%!important}\n .xposim-explorer .xposim-plane-label {grid-column:1/-1;grid-row:3;width:auto!important;justify-self:end}\n}\n\n.xposim-explorer .xposim-plot-compact {display:none!important}\n@container xp-explorer (max-width:540px) {\n .xposim-explorer .xposim-plot-wide {display:none!important}\n .xposim-explorer .xposim-plot-compact {display:block!important}\n}\n</style>\n'
UI_CSS=UI_CSS.replace('</style>', '''
.xp-parameter-overview,.xp-geometry-fields,.xp-material-fields {
 grid-template-columns:minmax(0,1.55fr) minmax(0,1fr);gap:9px;align-items:start;
}
.xp-geometry-fields {grid-template-columns:minmax(0,1.22fr) minmax(0,1fr)}
.xposim-parameters .xp-parameter-overview>div {height:100%}
.xposim-parameters>.xposim-section + .xposim-section {margin-top:7px}
.xposim-parameters .xp-detector-toggle {min-height:30px;margin:4px 0}
.xposim-parameters .xp-detector-toggle label {white-space:normal}
.xposim-parameters .xposim-field-caption {min-height:22px!important}
.xposim-parameters .xp-source-grid {grid-template-columns:repeat(4,minmax(0,1fr))}
.xposim-parameters .xp-source-grid .xposim-field-source {grid-column:span 2}
.xposim-parameters .xp-geometry-fields .xp-control-grid {grid-template-columns:repeat(2,minmax(0,1fr))}
.xposim-parameters .xp-parameter-overview .xposim-analytical-list,
.xposim-parameters .xp-material-fields .xposim-analytical-list {margin:0}
.xposim-parameters .xp-parameter-overview .xposim-analytical-list dd,
.xposim-parameters .xp-material-fields .xposim-analytical-list dd {font-size:15px}
.xposim-parameters .xp-parameter-overview .xposim-analytical-list .xposim-magnification {grid-template-columns:minmax(0,1fr) auto}
.xposim-parameters .xp-parameter-overview .xposim-analytic-status {min-height:23px!important}
.xp-optical-layout,.xp-element-view {min-width:0;background:var(--xp-bg);border:1px solid var(--xp-grid);border-radius:6px}
.xp-optical-layout svg,.xp-element-view svg {display:block;width:100%;height:auto}
.xp-element-view svg {max-height:224px}
.xp-optical-layout text,.xp-element-view text {fill:var(--xp-fg);font-family:inherit;dominant-baseline:alphabetic}
.xp-layout-label {font-size:17px;font-weight:650}
.xp-optical-layout .xp-layout-small,.xp-element-view .xp-layout-small {font-size:14px;fill:var(--xp-secondary)}
.xp-layout-axis,.xp-layout-dimension,.xp-layout-radius {fill:none;stroke:var(--xp-secondary);stroke-width:1.15}
.xp-layout-axis {stroke-dasharray:4 3;opacity:.65}
.xp-layout-radius {stroke-dasharray:3 2}
.xp-layout-source-shape {fill:#e5a344;fill-opacity:.13;stroke:#b47527;stroke-width:1.35}
.xp-layout-ray {fill:none;stroke:#d19843;stroke-width:1.65;stroke-linejoin:round}
.xp-layout-source {fill:#d19843;stroke:var(--xp-surface);stroke-width:1.4}
.xp-layout-lens,.xp-layout-disc {fill:#647da4;fill-opacity:.26;stroke:#6581ad;stroke-width:1.35}
.xp-layout-disc {fill-opacity:.16}
.xp-layout-focus,.xp-layout-focus-line {stroke:light-dark(#167c74,#4ec9b8);fill:light-dark(#167c74,#4ec9b8);stroke-width:2.5}
.xp-layout-caption {font-size:12px;color:var(--xp-secondary);padding:5px 0 0}
@container xp-parameters (max-width:780px) {
 .xp-parameter-overview,.xp-geometry-fields,.xp-material-fields {grid-template-columns:1fr}
 .xp-element-view {max-width:520px}
 .xp-optical-layout .xp-layout-label {font-size:20px}
 .xp-optical-layout .xp-layout-small {font-size:15px}
}
@container xp-section (max-width:360px) {
 .xposim-parameters .xp-parameter-overview .xposim-analytical-list .xposim-magnification {grid-template-columns:1fr}
}
</style>''')
UI_CSS=UI_CSS.replace('</style>', '''.xp-layout-slab {fill:var(--xp-secondary);fill-opacity:.15;stroke:var(--xp-secondary);stroke-width:1.35}
.xp-layout-cavity,.xp-layout-hidden {fill:none;stroke:var(--xp-secondary);stroke-width:1.25}
.xp-layout-hidden {stroke-dasharray:3 2;opacity:.43}
.xp-crl-hatch {fill:url(#xp-crl-hatch);stroke:none}
.xp-dimension-symbol {font-weight:650}
.xp-layout-hatch {fill:none;stroke:var(--xp-secondary);stroke-width:.6;opacity:.34}
.xp-layout-source-shape {fill:url(#xp-gaussian);fill-opacity:1;stroke:#b47caa;stroke-width:1}
</style>''')
UI_CSS=UI_CSS.replace('</style>', '''
.xposim-parameters .xp-parameter-overview .xposim-analytical-list>div {
 grid-template-columns:minmax(0,1fr) minmax(116px,auto);gap:3px 8px;
}
.xposim-parameters .xp-parameter-overview .xposim-analytical-list .xposim-magnification {grid-template-columns:1fr}
.xposim-parameters .xposim-magnification-axis {
 display:grid;grid-template-columns:minmax(0,1fr) auto;align-items:baseline;gap:4px 10px;text-align:right
}
.xposim-parameters .xposim-magnification-axis .xposim-axis-name {white-space:normal}
.xposim-parameters .xposim-magnification-axis .xposim-number {grid-column:2}
.xposim-parameters .xposim-magnification dd {padding:3px 0 2px 9px;border-left:2px solid var(--xp-grid)}
.xposim-parameters .xp-optical-layout svg {max-height:234px}
@container xp-parameters (max-width:620px) {
 .xposim-parameters .xp-control-grid {grid-template-columns:repeat(2,minmax(0,1fr))!important}
 .xp-control-grid .xposim-field-source,.xp-control-grid .xposim-field-source_method,
 .xp-control-grid .xposim-field-focusing,.xp-control-grid .xposim-field-material_mode,
 .xp-control-grid .xposim-field-material,.xp-control-grid .xposim-field-roi_mode {grid-column:1/-1}
}
</style>''')
ui_style=widgets.HTML(UI_CSS,layout=widgets.Layout(height='0px',margin='0',overflow='visible'))
parameter_editor=widgets.VBox([ui_style,source_section,geometry_section,material_section,detector_toggle,roi_section,overview],
    layout=widgets.Layout(width='100%',max_width='1280px',margin='0 auto',gap='8px',overflow='visible'))
if 'current_theme' in globals():parameter_editor.add_class('xposim-'+current_theme)
parameter_editor.add_class('xposim-parameters')
display(parameter_editor)


In [ ]:
#@title 3 · Results — press Compute after entering parameters above
import base64
import gc
import io
import time
import tempfile
import textwrap
from datetime import datetime, timezone
from pathlib import Path
from contextlib import ExitStack
from matplotlib.ticker import FuncFormatter, MaxNLocator
from matplotlib.colors import Normalize
from matplotlib.colorbar import ColorbarBase
from matplotlib import patheffects, font_manager

if 'controls' not in globals() or 'parameter_editor' not in globals():
    raise RuntimeError('Run the CRL and source parameters cell above before Results.')

# Reinitialization preserves inputs and waits for an explicit calculation.
_previous_theme=globals().get('theme_selector')
_theme_choice=_previous_theme.value if _previous_theme is not None else 'Auto'
_previous_state=globals().get('state',{})
if _previous_state.get('plane_update_handle') is not None:
    _previous_state['plane_update_handle'].cancel()
for name in ('slider','z_input','previous_button','next_button','peak_button','plane_label'):
    widget=_previous_state.get(name)
    if widget is not None:
        widget.unobserve_all()
        widget.close()
for name in ('explorer_root','compute_button','plane_image','scan_image','plane_compact_image','scan_compact_image','plane_view','progress_view','shared_scale','color_scale','results_notice','result_style','theme_selector','plane_heading','plane_metrics','scan_heading','source_comparison','report_button','report_message','report_panel'):
    old=globals().get(name)
    if old is not None:
        old.close()
state={'result':None,'slider':None,'handler':None,'plane_updates':0,'error':None,
       'z_unit_scale':1e-3,'scan_axes':None}
CMAP=plt.get_cmap('CMRmap')
_available_fonts={font.name for font in font_manager.fontManager.ttflist}
PLOT_FONT=next(name for name in ('Consolas','DejaVu Sans Mono','Liberation Mono') if name in _available_fonts)
INTENSITY_COLOR='#355f85'
PROFILE_COLOR='#355f85'
PROFILE_COLORS=(PROFILE_COLOR,'#788494')
WIDTH_COLORS=('#3d7665','#7aa494')
CURSOR_COLOR='#1ad1c1'
PANEL_SIZE=(6.4,5.45)
PLANE_SIZE=PANEL_SIZE
THEMES={
    'light':dict(surface='#ffffff',background='#f3f6fa',foreground='#172235',secondary='#465b74',border='#9aaabe',grid='#d6dee9',blue='#355f85',green='#3d7665',other='#788494',cursor='#119f93'),
    'dark':dict(surface='#222a35',background='#181d25',foreground='#f1f5fa',secondary='#c0ccdd',border='#64768f',grid='#3c495c',blue='#93c4ff',green='#91ddbb',other='#b5becd',cursor='#58e5d6')}
current_theme='light'
palette=THEMES[current_theme]
theme_selector=widgets.Dropdown(options=['Auto','Dark','Light'],value=_theme_choice,description='Theme',
    style={'description_width':'initial'},layout=widgets.Layout(width='160px',min_width='0'))
theme_selector.add_class('xposim-theme-selector')
compute_button=widgets.Button(description='Compute',button_style='primary',icon='play',
                              layout=widgets.Layout(width='125px'))
compute_button.style.button_color=INTENSITY_COLOR
report_button=widgets.Button(description='Download PDF report',icon='download',disabled=True,
    layout=widgets.Layout(width='auto',min_width='185px'))
report_button.tooltip='Complete a calculation to export a report.'
report_message=widgets.HTML(layout=widgets.Layout(min_width='0'))
report_panel=widgets.HBox([report_button,report_message],layout=widgets.Layout(
    width='100%',flex_flow='row wrap',align_items='center',gap='8px',margin='6px 0 0'))
report_panel.add_class('xposim-report-panel')
status=widgets.HTML('<span style="font-size:13px">Set parameters, then press Compute.</span>',layout=widgets.Layout(flex='1 1 0%',min_width='0'))
status.add_class('xposim-runtime')
summary=widgets.HTML(layout=widgets.Layout(width='100%',overflow='visible',margin='0'))
source_comparison=widgets.HTML(layout=widgets.Layout(width='100%',display='none'))
source_comparison.add_class('xposim-source-comparison')
results_notice=widgets.HTML(layout=widgets.Layout(width='100%',display='none',overflow='visible',margin='0'))
progress_view=widgets.HTML(layout=widgets.Layout(width='100%',display='none'))
progress_view.add_class('xposim-progress')
scan_image=widgets.Image(format='png',width='896',height='763',layout=widgets.Layout(width='100%',max_width='100%',min_width='0',height='auto',margin='0'))
plane_image=widgets.Image(format='png',width='896',height='763',layout=widgets.Layout(width='100%',max_width='100%',min_width='0',height='auto',margin='0'))
scan_compact_image=widgets.Image(format='png',width='504',height='1176',layout=widgets.Layout(width='100%',max_width='100%',height='auto',margin='0'))
plane_compact_image=widgets.Image(format='png',width='504',height='700',layout=widgets.Layout(width='100%',max_width='100%',height='auto',margin='0'))
for widget in (scan_image,plane_image):widget.add_class('xposim-plot-wide')
for widget in (scan_compact_image,plane_compact_image):widget.add_class('xposim-plot-compact')
plane_controls=widgets.VBox(layout=widgets.Layout(width='100%',display='none'))
details_text=widgets.HTML(layout=widgets.Layout(width='100%',overflow='visible',margin='0'))
plane_heading=widgets.HTML(layout=widgets.Layout(width='100%',overflow='visible',margin='0'))
scan_heading=widgets.HTML('<div class="xposim-panel-heading">Propagation along z</div>',layout=widgets.Layout(width='100%',overflow='visible',margin='0'))
plane_metrics=widgets.HTML(layout=widgets.Layout(display='none'))
plane_metrics.add_class('xposim-plane-readouts')
details=widgets.Accordion(children=[details_text],selected_index=None)
details.set_title(0,'Model, mesh and diagnostics')


def compact_number(value):
    text=f'{float(value):.2f}'.rstrip('0').rstrip('.')
    return '0' if text=='-0' else text


TICK_FORMATTER=FuncFormatter(lambda value,position:compact_number(value))


def engineering(values, unit=''):
    """Keep ticks compact and move a common power of ten to the axis label."""
    finite=np.asarray(values,dtype=float)
    finite=finite[np.isfinite(finite)]
    maximum=float(np.max(abs(finite))) if finite.size else 0.0
    exponent=0
    if maximum>=80:
        exponent=int(np.ceil(np.log10(maximum/80)))
    elif 0<maximum<0.01:
        exponent=int(np.floor(np.log10(maximum)))
    multiplier=10.0**exponent
    suffix=(' '+r'$\times\,10^{'+str(exponent)+'}$') if exponent else ''
    return multiplier,unit+suffix


def length_units(values):
    finite=np.asarray(values,dtype=float)
    finite=finite[np.isfinite(finite)]
    maximum=float(np.max(abs(finite))) if finite.size else 0
    scale,label=(1.0,'m') if maximum>=0.1 else ((1e-3,'mm') if maximum>=1e-4 else ((1e-6,'µm') if maximum>=1e-7 else (1e-9,'nm')))
    power,suffix=engineering(finite/scale,label)
    return scale*power,suffix


def fixed_axes(ax, x=True, y=True):
    for axis,enabled in ((ax.xaxis,x),(ax.yaxis,y)):
        if enabled:
            axis.set_major_locator(MaxNLocator(nbins=4,min_n_ticks=3))
            axis.set_major_formatter(TICK_FORMATTER)
    ax.tick_params(labelsize=10.5)
    for spine in ax.spines.values():
        spine.set_color(palette['border'])
    ax.set_facecolor(palette['surface'])


def set_render_value(widget,value):
    """Stage one complete frame before publishing any of its widgets."""
    frame=state.get('render_values')
    if frame is None:widget.value=value
    else:frame[widget]=value


def plot_png(figure,compact_widget,compact_size):
    stream=io.BytesIO()
    figure.savefig(stream,format='png',dpi=140,facecolor=palette['surface'])
    wide_size=figure.get_size_inches().copy()
    figure.set_size_inches(*compact_size)
    compact=io.BytesIO()
    compact_titles=[]
    for ax in figure.axes:
        if ax.get_title()=='Intensity distribution near the focus':
            compact_titles.append((ax,ax.get_title()))
            ax.set_title('Intensity distribution\nnear the focus')
    # Keep numeric extrema inside narrow axes without rounding their values.
    figure.canvas.draw()
    renderer=figure.canvas.get_renderer()
    label_positions=[]
    for ax in figure.axes:
        for artist in ax.texts:
            if artist.get_gid()!='xposim-extremum':continue
            bounds=artist.get_bbox_patch().get_window_extent(renderer)
            shift=min(0,ax.bbox.x1-4-bounds.x1)+max(0,ax.bbox.x0+4-bounds.x0)
            if shift:
                x,y=artist.get_position()
                label_positions.append((artist,(x,y)))
                artist.set_position((x+shift*72/figure.dpi,y))
    figure.savefig(compact,format='png',dpi=140,facecolor=palette['surface'])
    set_render_value(compact_widget,compact.getvalue())
    for ax,title in compact_titles:ax.set_title(title)
    for artist,position in label_positions:artist.set_position(position)
    figure.set_size_inches(*wide_size)
    figure.canvas.draw()
    plt.close(figure)
    return stream.getvalue()


def metric_value(value, unit=''):
    if value is None or not np.isfinite(value):
        return 'unresolved'
    exponent=0
    if value and (abs(value)<0.1 or abs(value)>=1e4):
        exponent=int(np.floor(np.log10(abs(value))))
    text=compact_number(value/10.0**exponent)
    if exponent:
        text+=f' × 10<sup>{exponent}</sup>'
    return text+((' '+html.escape(unit)) if unit else '')


def distance_value(value):
    # Cards use physical units instead of embedding Matplotlib math markup.
    scale,unit=(1.,'m') if abs(value)>=.1 else ((1e-3,'mm') if abs(value)>=1e-4 else ((1e-6,'µm') if abs(value)>=1e-7 else (1e-9,'nm')))
    return metric_value(value/scale,unit)


def width_value(value):
    if value is None or not np.isfinite(value):
        return 'unresolved'
    scale,unit=(1e-9,'nm') if abs(value)<1e-6 else (1e-6,'µm')
    return metric_value(value/scale,unit)


def axis_values(values, formatter):
    values=tuple(values)
    if len(values)==1:
        return formatter(values[0])
    return ' · '.join('<span style="white-space:nowrap">'+axis+': '+formatter(value)+'</span>' for axis,value in zip(('x','y'),values))


def _detect_theme():
    if colab_output is not None:
        try:
            detected=colab_output.eval_js("""(() => {
              const root=document.documentElement, body=document.body;
              const attr=(root.getAttribute('theme')||body.getAttribute('theme')||'').toLowerCase();
              if(attr==='dark'||attr==='light')return attr;
              if(root.classList.contains('dark')||body.classList.contains('dark'))return 'dark';
              const css=getComputedStyle(body);
              const primary=css.getPropertyValue('--colab-primary-text-color').trim();
              const probe=document.createElement('span');probe.style.color=primary||css.color;body.appendChild(probe);
              const resolved=getComputedStyle(probe).color;probe.remove();
              const colors=resolved.match(/[0-9.]+/g);
              if(colors&&colors.length>=3){
                const luminance=.2126*Number(colors[0])+.7152*Number(colors[1])+.0722*Number(colors[2]);
                return luminance>150?'dark':'light';
              }
              return matchMedia('(prefers-color-scheme:dark)').matches?'dark':'light';
            })()""",timeout_sec=2)
            if detected in THEMES:return detected
        except Exception:
            pass
    return current_theme


def plot_style():
    return {'savefig.bbox':None,'font.family':'monospace','font.monospace':[PLOT_FONT],
        'mathtext.fontset':'custom','mathtext.rm':PLOT_FONT,
        'mathtext.it':PLOT_FONT+':italic','mathtext.bf':PLOT_FONT+':bold',
        'mathtext.cal':PLOT_FONT,'mathtext.sf':PLOT_FONT,'mathtext.tt':PLOT_FONT,
        'font.size':11,'axes.titlesize':12,'axes.labelsize':11,
        'figure.facecolor':palette['surface'],'axes.facecolor':palette['surface'],
        'text.color':palette['foreground'],'axes.labelcolor':palette['foreground'],
        'xtick.color':palette['secondary'],'ytick.color':palette['secondary'],
        'grid.color':palette['grid'],'axes.edgecolor':palette['border']}


def set_plot_palette(theme):
    global palette,PROFILE_COLOR,PROFILE_COLORS,WIDTH_COLORS,INTENSITY_COLOR,CURSOR_COLOR
    palette=THEMES[theme]
    PROFILE_COLOR=INTENSITY_COLOR=palette['blue']
    PROFILE_COLORS=(palette['blue'],palette['other'])
    WIDTH_COLORS=(palette['green'],palette['other']);CURSOR_COLOR=palette['cursor']


def apply_theme(refresh=True):
    global current_theme
    chosen=_detect_theme() if theme_selector.value=='Auto' else theme_selector.value.lower()
    changed=chosen!=current_theme
    current_theme=chosen
    set_plot_palette(chosen)
    for widget in (parameter_editor,globals().get('explorer_root')):
        if widget is not None:
            widget.remove_class('xposim-light');widget.remove_class('xposim-dark')
            widget.add_class('xposim-'+current_theme)
    if state.get('slider') is not None:state['slider'].style.handle_color=CURSOR_COLOR
    if changed:state.pop('scan_cursor_m',None)
    if refresh and state.get('result') is not None:
        pending=state.get('pending_selection')
        target=pending[1] if pending is not None and pending[0] is state['result'] else state['slider'].value
        select_plane(target)


def profile_is_dashed(axis,result=None):
    if result is None:result=state.get('result')
    one_axis_2d=(result is not None and result.configuration.grid_dimension==2
        and result.configuration.geometry=='one-dimensional')
    return False if one_axis_2d else axis=='y'


def axis_badge(axis):
    color=PROFILE_COLORS[0 if axis=='x' else 1]
    line='dashed' if profile_is_dashed(axis) else 'solid'
    return '<span class="xposim-axis-badge"><span aria-hidden="true" style="border-top:2px '+line+' '+color+'"></span>'+axis+'</span>'


def axis_table(headers,rows):
    head='<thead><tr>'+''.join('<th scope="col">'+html.escape(label)+'</th>' for label in headers)+'</tr></thead>' if headers else ''
    body='<tbody>'+''.join('<tr>'+''.join('<td>'+value+'</td>' for value in row)+'</tr>' for row in rows)+'</tbody>'
    return '<table class="xposim-axis-table">'+head+body+'</table>'


def metric_box(label,unit,value):
    return '<section class="xposim-metric-box"><div class="xposim-metric-title">'+html.escape(label)+'<span class="xposim-unit">'+html.escape(unit)+'</span></div><div class="xposim-metric-value">'+value+'</div></section>'


def plain_number(value,digits=8):
    if value is None or not np.isfinite(value):return 'unresolved'
    text=f'{value:.{digits}f}'.rstrip('0').rstrip('.')
    return '0' if text=='-0' else text


def z_number(value,scale):
    step=state['result'].scan_step_m/scale
    digits=max(2,min(9,int(max(0,-np.floor(np.log10(step))))+2))
    return plain_number(value/scale,digits)


def z_text(value):
    return z_number(value,state['z_unit_scale'])


def depth_label(result):
    widths=result.depth_of_focus_m
    labels=['unresolved' if value is None else compact_number(value*1e3)+' mm' for value in widths]
    if len(labels)==1 or labels[0]==labels[-1]:
        return labels[0]
    return ' · '.join(axis+' '+label for axis,label in zip(result.fwhm_axes,labels))


def source_width_comparison(result):
    point=result.point_source_fwhm_at_peak_m
    if point is None:
        return ''
    rows=[]
    for axis,initial,final,factor in zip(result.fwhm_axes,point,np.atleast_1d(result.fwhm_at_peak_m),result.source_broadening_factor):
        detail='Unresolved' if factor is None else width_value(initial)+' → '+width_value(final)+' · × '+compact_number(factor)
        rows.append('<span><b>'+axis+'</b> '+detail+'</span>')
    return ('<div class="xp-source-widths"><b>Source broadening at the numerical reference</b>'
            '<span>Point → Gaussian FWHM</span>'+''.join(rows)+'</div>')


def update_summary(result,plane):
    aperture_size='diameter' if result.configuration.geometry=='two-dimensional' else 'width'
    headers=[('Numerical peak intensity distance','mm'),('Transmission','%'),
             ('Effective aperture '+aperture_size,'µm')]
    values=[z_number(result.peak_z_m,1e-3),compact_number(result.transmission*100),
            compact_number(result.effective_aperture_m*1e6)]
    note=' · central point reference' if result.configuration.source_type=='gaussian' else ''
    if result.configuration.detector_roi_m is not None:note+=' · manual calculation region'
    set_render_value(summary,'<section class="xposim-run"><div class="xposim-group-title">CRL · whole calculation'+note+'</div><dl class="xposim-run-metrics" style="--xp-metric-count:'+str(len(headers))+'">'+''.join('<div><dt>'+label+'</dt><dd><span class="xposim-metric-number">'+value+'</span><span class="xposim-unit">'+unit+'</span></dd></div>' for (label,unit),value in zip(headers,values))+'</dl></section>')
    set_render_value(scan_heading,'<div class="xposim-panel-heading">Propagation along z'
        '<span class="xp-depth-readout" title="The band follows the width threshold √2 × minimum FWHM.">Depth of focus · '
        +depth_label(result)+'</span></div>')
    set_render_value(plane_heading,'<div class="xposim-panel-heading">Selected transverse plane<span class="xposim-selected-z">z = '+z_text(plane.z_m)+' <span style="font-size:13px;font-weight:400">'+html.escape(state['z_unit'])+'</span></span></div>')
    set_render_value(plane_metrics,'')


def outside_legend(ax):
    # Constrained layout reserves this space, keeping keys clear of every curve.
    return ax.legend(frameon=False,fontsize=10,loc='lower left',
        bbox_to_anchor=(0,1.01),borderaxespad=0,ncol=2,handlelength=2,columnspacing=1.4)


def finite_extreme(values,mode,reference_index):
    values=np.asarray(values,dtype=float)
    indices=np.flatnonzero(np.isfinite(values))
    if not indices.size:return None
    best=np.max(values[indices]) if mode=='max' else np.min(values[indices])
    ties=indices[values[indices]==best]
    return int(ties[np.argmin(abs(ties-reference_index))])


def mark_scan_extrema(result,peak_ax,width_ax,width_arrays,zscale,peakscale,widthscale):
    extrema=[]
    index=finite_extreme(result.peak_intensity,'max',result.peak_index)
    if index is not None:
        x,y=result.z_m[index]/zscale,result.peak_intensity[index]/peakscale
        offset=-7 if x>np.mean(result.z_m)/zscale else 7
        label_artist=peak_ax.annotate('max: '+plot_metric(result.peak_intensity[index],'a.u.'),xy=(x,y),xytext=(offset,-8),textcoords='offset points',
            ha='right' if offset<0 else 'left',va='top',fontsize=9,color=palette['foreground'],
            bbox=dict(facecolor=palette['surface'],alpha=.9,edgecolor='none',pad=1))
        label_artist.set_gid('xposim-extremum');label_artist.set_in_layout(False)
        extrema.append(dict(kind='max',axis='intensity',index=index,z_m=float(result.z_m[index]),value=float(result.peak_intensity[index])))
    widths=[]
    for axis,values in zip(('x','y'),width_arrays):
        index=finite_extreme(values,'min',result.peak_index)
        if index is not None:widths.append((axis,index,float(values[index])))
    coincident=(len(widths)==2 and widths[0][1]==widths[1][1]
        and np.isclose(widths[0][2],widths[1][2],rtol=1e-6,atol=0))
    for n,(axis,index,value) in enumerate(widths):
        extrema.append(dict(kind='min',axis=axis,index=index,z_m=float(result.z_m[index]),value=value))
        if coincident and n==1:continue
        x,y=result.z_m[index]/zscale,value/widthscale
        label='min x,y' if coincident else ('min '+axis if len(width_arrays)>1 else 'min')
        unit_scale,unit=(1e-9,'nm') if value<1e-6 else (1e-6,'µm')
        label+=': '+plot_metric(value/unit_scale,unit)
        offset=-7 if x>np.mean(result.z_m)/zscale else 7
        label_artist=width_ax.annotate(label,xy=(x,y),xytext=(offset,6+n*12),textcoords='offset points',
            ha='right' if offset<0 else 'left',va='bottom',fontsize=9,color=palette['foreground'],
            bbox=dict(facecolor=palette['surface'],alpha=.9,edgecolor='none',pad=1))
        label_artist.set_gid('xposim-extremum');label_artist.set_in_layout(False)
    return extrema


def render_scan(result, selected_plane):
    zscale,zunit=length_units(result.z_m)
    xscale,xunit=length_units(result.x_m)
    peakscale,peakunit=engineering(result.peak_intensity,'a.u.')
    xwidth=result.fwhm_m if result.fwhm_m.ndim==1 else result.fwhm_m[:,0]
    width_arrays=[xwidth]
    if result.configuration.geometry=='two-dimensional':
        width_arrays.append(result.fwhm_m[:,1])
    widthscale,widthunit=length_units(np.concatenate(width_arrays))
    with plt.rc_context(plot_style()):
        figure=plt.figure(figsize=PANEL_SIZE,layout='constrained')
        spec=figure.add_gridspec(3,1,height_ratios=[2.0,1,1],hspace=.04)
        map_ax=figure.add_subplot(spec[0,0])
        peak_ax=figure.add_subplot(spec[1,0],sharex=map_ax)
        width_ax=figure.add_subplot(spec[2,0],sharex=map_ax)
        top=max(result.intensity_reference,np.finfo(float).tiny)
        mesh=map_ax.pcolormesh(result.z_m/zscale,result.x_m/xscale,
            result.longitudinal_intensity.T/top,shading='auto',cmap=CMAP,norm=Normalize(0,1),rasterized=True)
        manual_roi=result.configuration.detector_roi_m
        map_ax.set(ylabel=f'x ({xunit})',
                   title='Intensity in sampled region' if manual_roi is not None else 'Intensity distribution near the focus')
        if manual_roi is not None:
            x_bounds=manual_roi if result.configuration.grid_dimension==1 else manual_roi[0]
            map_ax.set_ylim(*(value/xscale for value in x_bounds))
            if result.configuration.scan_range_m is not None:
                map_ax.set_xlim(*(value/zscale for value in result.configuration.scan_range_m))
        if result.configuration.grid_dimension==1:
            cax=map_ax.inset_axes([1.025,0,.025,1])
            cb=figure.colorbar(mesh,cax=cax);cb.set_ticks([0,.5,1])
            cb.ax.yaxis.set_major_formatter(TICK_FORMATTER);cb.ax.tick_params(labelsize=9,colors=palette['secondary'])
            cb.outline.set_visible(False);cb.ax.set_title('I / Iref',fontsize=9,color=palette['foreground'])
        peak_ax.plot(result.z_m/zscale,result.peak_intensity/peakscale,color=INTENSITY_COLOR,lw=1.7)
        peak_ax.set(ylabel='Peak I\n('+peakunit+')')
        for index,width in enumerate(width_arrays):
            width_ax.plot(result.z_m/zscale,width/widthscale,label='x' if index==0 else 'y',
                          color=WIDTH_COLORS[index],lw=2.3 if index==0 else 1.7,
                          ls='-' if index==0 else (0,(4,2.5)),alpha=.85 if index==0 else .95)
        last_band=None
        for index,bounds in enumerate(result.depth_of_focus_bounds_m):
            if bounds is None or (last_band is not None and np.allclose(bounds,last_band,rtol=1e-8,atol=0)):
                continue
            width_ax.axvspan(bounds[0]/zscale,bounds[1]/zscale,color=WIDTH_COLORS[index],
                             alpha=.18,linewidth=0,zorder=0)
            last_band=bounds
        if len(width_arrays)>1:
            outside_legend(width_ax)
        width_label=(r'$\mathrm{FWHM}_x$' if len(width_arrays)==1 else r'$\mathrm{FWHM}_{x,y}$')
        width_ax.set(xlabel=f'z from CRL exit ({zunit})',ylabel=width_label+' ('+widthunit+')')
        current_z=selected_plane.z_m
        for ax in (map_ax,peak_ax,width_ax):
            fixed_axes(ax)
            ax.axvline(result.peak_z_m/zscale,color=palette['secondary'],ls=':',lw=.85)
            ax.axvline(current_z/zscale,color=CURSOR_COLOR,lw=1.05,
                         path_effects=[patheffects.Stroke(linewidth=2.0,foreground=palette['surface'],alpha=.45),patheffects.Normal()])
        peak_ax.plot([current_z/zscale],[selected_plane.metrics.max_intensity/peakscale],'o',color=CURSOR_COLOR,ms=4,mec=palette['foreground'],mew=.45)
        for width in np.atleast_1d(selected_plane.metrics.fwhm)[:len(width_arrays)]:
            if width is not None and np.isfinite(width):
                width_ax.plot([current_z/zscale],[width/widthscale],'o',color=CURSOR_COLOR,ms=4,mec=palette['foreground'],mew=.45)
        extrema=mark_scan_extrema(result,peak_ax,width_ax,width_arrays,zscale,peakscale,widthscale)
        for ax in (map_ax,peak_ax):
            ax.tick_params(labelbottom=False)
        for ax in (peak_ax,width_ax):
            ax.grid(alpha=.5)
        state['scan_extrema']=extrema
        state['scan_axes']=(map_ax,peak_ax,width_ax)
        set_render_value(scan_image,plot_png(figure,scan_compact_image,(3.6,8.4)))
        state['scan_cursor_m']=current_z
        state['scan_render_key']=(result,current_theme,current_z)


def nearest_z(z_m):
    """Snap physical z to the regular computed overview grid."""
    positions=state['result'].z_m
    return float(positions[np.argmin(abs(positions-float(z_m)))])


_MISSING_RENDER_STATE=object()
_RENDER_STATE_KEYS=('last_plane','plane_updates','scan_axes','scan_cursor_m',
    'scan_render_key','plane_axes','colorbar_axes','fwhm_annotations','map_fwhm_rulers',
    'map_fwhm_grouped','profile_peak','profile_fwhm_height','scan_extrema')


def _restore_render_state(snapshot):
    for key,value in snapshot.items():
        if value is _MISSING_RENDER_STATE:state.pop(key,None)
        else:state[key]=value


def _sync_navigation(index,result):
    state['syncing_navigation']=True
    try:
        state['slider'].value=index
        state['z_input'].value=z_text(result.z_m[index])
        state['previous_button'].disabled=index==0
        state['next_button'].disabled=index==len(result.z_m)-1
        state['plane_label'].value=f'<span style="font-size:12px">Plane {index+1} / {len(result.z_m)}</span>'
    finally:
        state['syncing_navigation']=False


def select_plane(index):
    """Render the latest request and commit one consistent plane/cursor frame.

    Reentrant input replaces the pending request instead of being dropped.
    Plot bytes and readouts are staged; a failed or superseded render cannot
    publish a partial frame or mark an old overview image as up to date.
    """
    result=state.get('result')
    if result is None:return
    index=int(np.clip(index,0,len(result.z_m)-1))
    state['pending_selection']=(result,index)
    if state.get('updating_selection'):return
    state['updating_selection']=True
    initial_result=result
    ready_progress=progress_view.value if state.get('progress_stage')=='ready' else None
    if ready_progress is not None:
        progress_view.value='<div style="display:flex;align-items:center;font-size:11px;margin:1px 0 3px"><span>Updating selected plane…</span></div>'
    try:
        while state.get('pending_selection') is not None:
            result,index=state['pending_selection']
            state['pending_selection']=None
            if state.get('result') is not result:continue
            snapshot={key:state.get(key,_MISSING_RENDER_STATE) for key in _RENDER_STATE_KEYS}
            frame={};state['render_values']=frame
            theme=current_theme
            try:
                _render_plane(float(result.z_m[index]))
                pending=state.get('pending_selection')
                if state.get('result') is not result:continue
                if (pending is not None and (pending[0] is not result or pending[1]!=index)) or theme!=current_theme:
                    _restore_render_state(snapshot)
                    if state.get('pending_selection') is None:state['pending_selection']=(result,index)
                    continue
                if pending is not None:state['pending_selection']=None
                # All expensive work has finished before controls/readouts change.
                navigation=[state[name] for name in ('slider','z_input','previous_button','next_button','plane_label')]
                with ExitStack() as stack:
                    for widget in (*frame,*navigation):stack.enter_context(widget.hold_sync())
                    for widget,value in frame.items():widget.value=value
                    _sync_navigation(index,result)
                    state['displayed_index']=index
                    state['frame_revision']=state.get('frame_revision',0)+1
                    state['error']=None
                if state.get('ready_status') and state.get('progress_stage')=='ready':status.value=state['ready_status']
            except Exception as exc:
                if state.get('result') is result:_restore_render_state(snapshot)
                if state.get('pending_selection') is not None:continue
                state['error']=exc
                previous=state.get('displayed_index')
                if previous is not None and state.get('result') is result:
                    _sync_navigation(previous,result)
                status.value='<span style="font-size:12px;color:#b45309">Plane update failed: '+html.escape(str(exc))+'</span>'
            finally:
                state.pop('render_values',None)
    finally:
        state['updating_selection']=False
        if ready_progress is not None and state.get('result') is initial_result and state.get('progress_stage')=='ready':
            progress_view.value=ready_progress


def draw_plane(z_m):
    """Use the same selection path for direct z requests and theme changes."""
    result=state.get('result')
    if result is not None:select_plane(int(np.argmin(abs(result.z_m-float(z_m)))))


def build_plane_controls(result,zscale,zunit):
    index=int(np.argmin(abs(result.z_m-result.peak_z_m)))
    slider=widgets.IntSlider(value=index,min=0,max=len(result.z_m)-1,step=1,
        description='',readout=False,continuous_update=True,behavior='drag-snap',
        layout=widgets.Layout(width='auto',flex='1 1 240px',min_width='180px'))
    z_input=widgets.Text(value=z_text(result.z_m[index]),
        description=f'z ({zunit})',continuous_update=False,
        style={'description_width':'initial'},layout=widgets.Layout(width='160px'))
    previous=widgets.Button(icon='chevron-left',tooltip='Previous z plane',layout=widgets.Layout(width='34px'))
    following=widgets.Button(icon='chevron-right',tooltip='Next z plane',layout=widgets.Layout(width='34px'))
    label=widgets.HTML(layout=widgets.Layout(width='105px'))
    slider.style.handle_color=CURSOR_COLOR
    reference=widgets.Button(description='Peak',icon='crosshairs',tooltip='Go to the exact numerical peak-intensity reference plane',layout=widgets.Layout(width='77px'))
    reference.add_class('xposim-peak-plane')
    slider.add_class('xposim-plane-slider')
    z_input.add_class('xposim-z-input')
    previous.add_class('xposim-previous-plane')
    following.add_class('xposim-next-plane')
    label.add_class('xposim-plane-label')
    state.update(slider=slider,z_input=z_input,previous_button=previous,
                 next_button=following,peak_button=reference,plane_label=label)
    def slider_changed(change):
        if state.get('syncing_navigation') or state.get('slider') is not slider or state.get('result') is not result:return
        select_plane(change['new'])
    def z_changed(change):
        if state.get('syncing_navigation') or state.get('z_input') is not z_input or state.get('result') is not result:return
        try:
            target=float(change['new'])*state['z_unit_scale']
            if not np.isfinite(target):
                raise ValueError('Enter a finite z value.')
            select_plane(int(np.argmin(abs(result.z_m-target))))
        except (ValueError,TypeError) as exc:
            status.value='<span style="font-size:12px;color:#b45309">'+html.escape(str(exc))+'</span>'
    slider.observe(slider_changed,names='value')
    z_input.observe(z_changed,names='value')
    def navigate(target):
        if state.get('slider') is slider and state.get('result') is result:select_plane(target)
    def step(delta):
        pending=state.get('pending_selection')
        base=pending[1] if pending is not None and pending[0] is result else slider.value
        navigate(base+delta)
    previous.on_click(lambda _:step(-1))
    following.on_click(lambda _:step(1))
    reference.on_click(lambda _:navigate(index))
    state['handler']=slider_changed
    plane_controls.children=(widgets.HBox([previous,slider,following,reference,z_input,label],
        layout=widgets.Layout(width='100%',align_items='center',flex_flow='row wrap')), )
    plane_controls.children[0].add_class('xposim-selection-row')
    select_plane(index)


def half_max_crossings(coordinate,values):
    values=np.asarray(values);coordinate=np.asarray(coordinate)
    i=int(np.argmax(values));half=values[i]/2
    left=np.flatnonzero(values[:i]<half)
    right=np.flatnonzero(values[i+1:]<half)+i+1
    if not left.size or not right.size or half<=0:return None
    l=int(left[-1]);r=int(right[0])
    def crossing(a,b):
        return float(coordinate[a]+(half-values[a])/(values[b]-values[a])*(coordinate[b]-coordinate[a]))
    return crossing(l,l+1),crossing(r-1,r)


def annotate_fwhm(ax,profiles,scale,axes,frame):
    xmin,xmax=ax.get_xlim();ymin,ymax=ax.get_ylim()
    span=xmax-xmin
    measurements=[]
    for index,(coord,profile,axis) in enumerate(profiles):
        if axis not in axes:continue
        crossing=half_max_crossings(coord,profile)
        label=r'$\mathrm{FWHM}_{'+axis+'}$: '
        if crossing is None:
            measurements.append((index,axis,None,label+'unresolved',0.0))
            continue
        lower,upper=np.asarray(crossing)/scale
        width=float(crossing[1]-crossing[0])
        unit_scale,unit=(1e-9,'nm') if width<1e-6 else (1e-6,'µm')
        label+=compact_number(width/unit_scale)+' '+unit
        axis_span=(ymax-ymin) if axis=='x' else span
        measurements.append((index,axis,crossing,label,(upper-lower)/axis_span))
    # Select one layout for all widths; do not mix a corner readout with a
    # rotated/ruler label. The axis label reserves space below the square map.
    grouped=any(crossing is None or fraction<.05 for _,_,crossing,_,fraction in measurements)
    annotations=[];rulers=[]
    for index,axis,crossing,label,fraction in measurements:
        if crossing is None:continue
        lower,upper=np.asarray(crossing)/scale
        color=PROFILE_COLORS[index]
        if not grouped:
            if axis=='y':
                y=ymin+.2*(ymax-ymin)
                ax.annotate('',xy=(upper,y),xytext=(lower,y),arrowprops=dict(arrowstyle='|-|',color=color,lw=1.1))
                if not grouped:
                    ax.text((lower+upper)/2,y-.06*span,label,ha='center',va='top',fontsize=10,color=palette['foreground'],
                        bbox=dict(facecolor=palette['surface'],alpha=.9,edgecolor='none',pad=2))
            else:
                x=xmin+.18*span
                ax.annotate('',xy=(x,upper),xytext=(x,lower),arrowprops=dict(arrowstyle='|-|',color=color,lw=1.1))
                if not grouped:
                    ax.text(x-.055*span,(lower+upper)/2,label,rotation=90,ha='right',va='center',fontsize=10,color=palette['foreground'],
                        bbox=dict(facecolor=palette['surface'],alpha=.9,edgecolor='none',pad=2))
            rulers.append(axis)
        annotations.append((axis,tuple(map(float,crossing))))
    if grouped:
        ax.set_xlabel(ax.get_xlabel()+'\n'+'\n'.join(label for _,_,_,label,_ in measurements))
        ax.xaxis.label.set_linespacing(1.3)
        ax.xaxis.label.set_gid('xposim-fwhm-group')
    frame['fwhm_annotations']=annotations
    frame['map_fwhm_rulers']=rulers
    frame['map_fwhm_grouped']=grouped


def annotate_map_peak(ax,plane):
    ax.text(.97,.97,'Peak intensity\n'+plot_metric(plane.metrics.max_intensity,'a.u.'),
        transform=ax.transAxes,ha='right',va='top',fontsize=10,
        color=palette['foreground'],
        bbox=dict(facecolor=palette['surface'],edgecolor=palette['border'],alpha=.92,pad=3))


def plot_metric(value,unit=''):
    if value is None or not np.isfinite(value):return 'unresolved'
    exponent=int(np.floor(np.log10(abs(value)))) if value and (abs(value)<.1 or abs(value)>=1e4) else 0
    number=compact_number(value/10.0**exponent)
    if exponent:number=r'$'+number+r'\times 10^{'+str(exponent)+'}$'
    return number+((' '+unit) if unit else '')


def annotate_profile(ax,coordinate,values,scale,reference,frame,show_peak=True):
    index=int(np.argmax(values))
    peak=float(values[index])/reference
    label='Peak intensity\n'+plot_metric(float(values[index]),'a.u.')
    if show_peak:ax.plot([coordinate[index]/scale],[peak],'o',ms=4,color=PROFILE_COLOR,zorder=5)
    if show_peak:ax.annotate(label,xy=(coordinate[index]/scale,peak),xytext=(.98,.93),
        textcoords='axes fraction',ha='right',va='top',fontsize=11,
        color=palette['foreground'],arrowprops=dict(arrowstyle='-',color=PROFILE_COLOR,lw=.9),
        bbox=dict(facecolor=palette['surface'],edgecolor=palette['border'],alpha=.95,pad=4))
    crossing=half_max_crossings(coordinate,values)
    frame['profile_peak']=(float(coordinate[index]),float(values[index])) if show_peak else None
    frame['fwhm_annotations']=[]
    if crossing is None:
        ax.text(.5,.72,r'$\mathrm{FWHM}_x$: unresolved',transform=ax.transAxes,ha='center',
            color=palette['foreground'],fontsize=11)
        return
    left,right=np.asarray(crossing)/scale
    height=peak/2
    ax.annotate('',xy=(right,height),xytext=(left,height),
        arrowprops=dict(arrowstyle='|-|',color=WIDTH_COLORS[0],lw=1.4))
    width=crossing[1]-crossing[0]
    unit_scale,unit=(1e-9,'nm') if width<1e-6 else (1e-6,'µm')
    label=r'$\mathrm{FWHM}_x$ = '+compact_number(width/unit_scale)+' '+unit
    ax.annotate(label,xy=((left+right)/2,height),xytext=(0,6),textcoords='offset points',
        ha='center',va='bottom',fontsize=11,color=palette['foreground'],
        bbox=dict(facecolor=palette['surface'],alpha=.9,edgecolor='none',pad=2))
    frame['fwhm_annotations']=[('x',tuple(map(float,crossing)))]
    frame['profile_fwhm_height']=height


def render_transverse(result,plane):
    frame=state
    peak=max(result.intensity_reference,np.finfo(float).tiny)
    with plt.rc_context(plot_style()):
        figure=plt.figure(figsize=PLANE_SIZE,layout='constrained')
        if plane.y_m is not None:
            spec=figure.add_gridspec(2,1,height_ratios=[3.2,1.2],hspace=.05)
            map_ax=figure.add_subplot(spec[0,0]);profile_ax=figure.add_subplot(spec[1,0])
            scale,unit=length_units(np.r_[plane.x_m,plane.y_m])
            j,i=np.unravel_index(int(np.argmax(plane.intensity)),plane.intensity.shape)
            profiles=[(plane.x_m,plane.intensity[j,:],'x'),(plane.y_m,plane.intensity[:,i],'y')]
            dx=result.detector_step_m;dy=result.detector_step_y_m
            span=max(float(np.ptp(plane.x_m)+dx),float(np.ptp(plane.y_m)+dy))/scale
            cx=float((plane.x_m[0]+plane.x_m[-1])/2)/scale
            cy=float((plane.y_m[0]+plane.y_m[-1])/2)/scale
            mesh=map_ax.pcolormesh(plane.y_m/scale,plane.x_m/scale,plane.intensity.T/peak,
                shading='auto',cmap=CMAP,norm=Normalize(0,1),rasterized=True)
            map_ax.set(xlabel=f'y ({unit})',ylabel=f'x ({unit})',title='Transverse intensity',
                xlim=(cy-span/2,cy+span/2),ylim=(cx-span/2,cx+span/2),aspect='equal')
            map_ax.set_facecolor(CMAP(0))
            fixed_axes(map_ax);map_ax.set_facecolor(CMAP(0))
            map_ax.yaxis.tick_right();map_ax.yaxis.set_label_position('right')
            cax=map_ax.inset_axes([-.13,0,.035,1])
            cb=figure.colorbar(mesh,cax=cax);cb.set_ticks([0,.5,1])
            cb.ax.yaxis.set_ticks_position('left');cb.ax.yaxis.set_major_formatter(TICK_FORMATTER)
            cb.ax.tick_params(labelsize=9,colors=palette['secondary']);cb.outline.set_visible(False)
            cb.ax.set_ylabel('I / Iref',fontsize=10,color=palette['foreground'])
            cb.ax.yaxis.set_label_position('right')
            frame['fwhm_annotations']=[];frame['map_fwhm_rulers']=[];frame['map_fwhm_grouped']=False
            if result.configuration.geometry=='two-dimensional':
                annotate_fwhm(map_ax,profiles,scale,result.fwhm_axes,frame)
            else:
                profiles=profiles[:1]
            annotate_map_peak(map_ax,plane)
            frame['plane_axes']=(map_ax,profile_ax);frame['colorbar_axes']=cax
        else:
            profile_ax=figure.add_subplot(111)
            profiles=[(plane.x_m,plane.intensity,'x')]
            frame['plane_axes']=(profile_ax,);frame['fwhm_annotations']=[]
        scale,unit=length_units(np.concatenate([coord for coord,profile,label in profiles]))
        for index,(coord,profile,label) in enumerate(profiles):
            legend_label=(label+(' (focusing)' if label=='x' else ' (diffraction)')
                if result.configuration.grid_dimension==2 and result.configuration.geometry=='one-dimensional' else label)
            profile_ax.plot(coord/scale,profile/peak,label=legend_label,
                color=PROFILE_COLORS[index],lw=2.3 if index==0 else 1.7,
                ls=(0,(4,2.5)) if profile_is_dashed(label,result) else '-',alpha=.85 if index==0 else .95,zorder=2+index)
        profile_ax.set(xlabel=(f'Transverse coordinate ({unit})' if len(profiles)>1 else f'x ({unit})'),ylabel='I / Iref',
            title='' if len(profiles)>1 else ('Focused x profile' if plane.y_m is not None else 'Transverse intensity'),ylim=(-.035,1.05))
        if len(profiles)>1:outside_legend(profile_ax)
        profile_ax.grid(alpha=.5);fixed_axes(profile_ax)
        if len(profiles)==1:annotate_profile(profile_ax,profiles[0][0],profiles[0][1],scale,peak,frame,show_peak=plane.y_m is None)
        set_render_value(plane_image,plot_png(figure,plane_compact_image,(3.6,5.0) if plane.y_m is not None else (3.6,4.2)))


def _render_plane(z_m):
    result=state['result']
    if result is None:return
    plane=result.plane_at(nearest_z(z_m))
    state['last_plane']=plane;state['plane_updates']+=1
    render_transverse(result,plane)
    scan_key=state.get('scan_render_key')
    if scan_key is None or scan_key[0] is not result or scan_key[1]!=current_theme or scan_key[2]!=plane.z_m:
        render_scan(result,plane)
    update_summary(result,plane)


STAGE_LABELS={'preparing':'Preparing calculation','grid':'Preparing CRL and grid',
    'source_preparation':'Preparing the full Gaussian source',
    'window_checks':'Checking propagation window','source_modes':'Preparing source modes',
    'detector':'Preparing transverse evaluation grid','peak_search':'Finding peak-intensity distance',
    'axial_roi':'Finding display interval','sampling':'Calculating z planes',
    'finalizing':'Finalizing beam metrics','complete':'Numerical calculation complete',
    'rendering':'Rendering plots','ready':'Complete','failed':'Calculation stopped'}


def duration_text(seconds):
    seconds=max(0,int(np.ceil(seconds)))
    minutes,seconds=divmod(seconds,60)
    hours,minutes=divmod(minutes,60)
    return (f'{hours:d}:{minutes:02d}:{seconds:02d}' if hours else f'{minutes:02d}:{seconds:02d}')


def show_progress(stage,completed=0,total=None,elapsed_s=0.0,eta_s=None,detail=''):
    label=STAGE_LABELS.get(stage,stage)
    if stage=='ready':
        progress_view.value='<div style="display:flex;align-items:center;gap:8px;font-size:11px;opacity:.8;margin:1px 0 3px"><progress max="1" value="1" aria-label="Complete" style="width:85px;height:8px;accent-color:#3d7665"></progress><span>Complete · '+duration_text(elapsed_s)+'</span></div>'
        progress_view.layout.display=''
        state['progress_stage']=stage
        return
    if total is None:
        amount=f'{completed} evaluations' if stage in {'peak_search','axial_roi'} else ''
        bar='<progress aria-label="'+html.escape(label,quote=True)+'" style="width:100%;height:12px;accent-color:#355f85"></progress>'
    else:
        amount=f'{completed} / {total}'
        bar='<progress aria-label="'+html.escape(label,quote=True)+'" max="'+str(total)+'" value="'+str(completed)+'" style="width:100%;height:12px;accent-color:#355f85"></progress>'
    if stage=='ready':
        remaining='Completed'
    elif stage=='failed':
        remaining='Stopped'
    elif eta_s is not None:
        remaining='ETA for this stage: ≈ '+duration_text(eta_s)
    elif total is None:
        remaining='ETA: estimating · adaptive stage' if stage in {'grid','peak_search','axial_roi'} else 'ETA: estimating'
    else:
        remaining='ETA for this stage: estimating'
    progress_view.value='<div style="padding:7px 10px;margin:3px 0 5px;border:1px solid rgba(127,127,127,.2);border-radius:8px;font-size:12px"><div style="display:flex;justify-content:space-between;gap:12px"><b>'+html.escape(label)+'</b><span>'+html.escape(amount)+'</span></div>'+bar+'<div style="display:flex;justify-content:space-between;gap:12px;flex-wrap:wrap"><span>Elapsed: '+duration_text(elapsed_s)+'</span><span>'+html.escape(remaining)+'</span></div>'+('<div style="font-size:11px;opacity:.75;margin-top:3px">'+html.escape(detail)+'</div>' if detail else '')+'</div>'
    progress_view.layout.display=''
    state['progress_stage']=stage


def on_calculation_progress(update):
    state['last_progress']=update
    show_progress(update.stage,update.completed,update.total,update.elapsed_s,update.eta_s,update.detail)


def compute_focus(_=None):
    started=time.perf_counter()
    apply_theme(refresh=False)
    show_progress('preparing')
    compute_button.disabled=True
    report_button.disabled=True;report_message.value=''
    status.value='<span style="font-size:12px"><progress style="width:90px;height:8px"></progress> Computing…</span>'
    state['error']=None
    for name in ('slider','z_input','previous_button','next_button','peak_button','plane_label'):
        widget=state.get(name)
        if widget is not None:
            widget.unobserve_all();widget.close()
        state[name]=None
    state['slider']=None;state['result']=None;state['pending_selection']=None
    state.pop('last_plane',None)
    state.pop('scan_cursor_m',None)
    state.pop('scan_render_key',None)
    state.pop('displayed_index',None)
    summary.value='';scan_image.value=b'';plane_image.value=b'';scan_compact_image.value=b'';plane_compact_image.value=b'';plane_metrics.value='';plane_heading.value=''
    results_panel.layout.display='none';details.layout.display='none';plane_controls.layout.display='none'
    source_comparison.value=''
    source_comparison.layout.display='none'
    gc.collect()
    try:
        config=read_configuration()
        result=xpos.estimate_crl_focus(config,progress=on_calculation_progress)
        elapsed=time.perf_counter()-started
        state['result']=result
        plane_compact_image.height='588' if result.configuration.grid_dimension==1 else '700'
        state['intensity_reference']=max(result.intensity_reference,np.finfo(float).tiny)
        show_progress('rendering',elapsed_s=time.perf_counter()-started)
        zscale,zunit=length_units(result.z_m)
        state['z_unit_scale']=zscale;state['z_unit']=zunit
        build_plane_controls(result,zscale,zunit)
        if state['error'] is not None:
            raise state['error']
        results_panel.layout.display='flex'
        plane_controls.layout.display=''
        details.layout.display=''
        source_comparison.value=source_width_comparison(result)
        source_comparison.layout.display='' if source_comparison.value else 'none'
        source_label=('Plane wave at CRL entrance' if config.source_type=='plane' else 'Point source' if config.source_type=='point' else 'Gaussian source · FWHM '+axis_values(
            (config.source_fwhm_x_m,) if config.grid_dimension==1 else (config.source_fwhm_x_m,config.source_fwhm_y_m),width_value))
        material_label=('Direct δ / β' if config.material_delta is not None else html.escape(config.material_formula)+' · '+metric_value(config.density_g_cm3,'g/cm³'))
        rows=[('Material',material_label),('δ',metric_value(result.delta)),('β',metric_value(result.beta)),
              ('Surface radius / full aperture',distance_value(config.radius_m)+' / '+distance_value(config.aperture_m)),
              ('Source',source_label),
              *([] if config.source_type=='plane' else [('Source-to-CRL distance',distance_value(config.source_distance_m))]),
              ('CRL length',distance_value(result.lens_length_m)),
              ('Propagation grid',str(result.grid_shape)),
              ('Transverse evaluation grid',str(result.detector_shape)),
              ('Propagation x step',distance_value(result.grid_step_m)),
              ('Evaluation x step',distance_value(result.detector_step_m)),
              ('Peak search reference','Plane wave' if config.source_type=='plane' else 'Central source point'),
              ('Peak search evaluations',str(result.numerical_peak.evaluations)),
              *([('Sampled full-source peak z',distance_value(result.sampled_peak_z_m))] if config.source_type=='gaussian' else [])]
        detail='<div style="font-size:12px;line-height:1.6">'+''.join('<div><b>'+label+':</b> '+value+'</div>' for label,value in rows)
        diagnostic_text=(message.partition(':')[2].strip().replace('detector ROI','calculation region')
                         .replace('detector sampling','transverse evaluation sampling') for message in result.diagnostics)
        detail+=''.join('<div>'+html.escape(message)+'</div>' for message in diagnostic_text)
        details_text.value=detail+'</div>'
        details.set_title(0,f'Model, mesh and diagnostics ({len(result.diagnostics)})')
        device='CPU / NumPy' if result.backend=='numpy' else _cuda_device_name()
        elapsed=time.perf_counter()-started
        status.value=runtime_grid_info(result,device,elapsed)
        state['ready_status']=status.value
        state['report_runtime']=(device,elapsed)
        report_button.disabled=False
        report_button.tooltip='Export a one-page summary with transverse sections before, at and after the peak reference.'
        show_progress('ready',1,1,time.perf_counter()-started,0.0)
        sync_results_notice()
    except Exception as exc:
        show_progress('failed',elapsed_s=time.perf_counter()-started,detail=str(exc))
        state['error']=exc;state['result']=None
        summary.value='';scan_image.value=b'';plane_image.value=b'';scan_compact_image.value=b'';plane_compact_image.value=b'';plane_metrics.value='';plane_heading.value=''
        plane_controls.children=()
        source_comparison.value='';source_comparison.layout.display='none'
        results_panel.layout.display='none';details.layout.display='none';plane_controls.layout.display='none'
        status.value='<span style="font-size:12px;color:#b45309">'+html.escape(str(exc))+'</span>'
    finally:
        compute_button.disabled=False
        if state['result'] is None:
            report_button.disabled=True
            report_button.tooltip='Complete a calculation to export a report.'


def runtime_grid_info(result,device,elapsed):
    def grid_item(label,shape,dx,dy):
        count=str(shape[-1]) if len(shape)==1 else str(shape[-1])+' × '+str(shape[0])
        text=label+': '+count+' · Δx '+width_value(dx)
        if dy is not None:text+=' · Δy '+width_value(dy)
        return '<span style="white-space:normal">'+text+'</span>'
    items=['<b>'+html.escape(device)+'</b>','<span>'+compact_number(elapsed)+' s</span>',
           grid_item('Propagation grid',result.grid_shape,result.grid_step_m,result.grid_step_y_m)]
    if result.detector_shape!=result.grid_shape or result.detector_step_m!=result.grid_step_m or result.detector_step_y_m!=result.grid_step_y_m:
        items.append(grid_item('Transverse evaluation grid',result.detector_shape,result.detector_step_m,result.detector_step_y_m))
    items.append('<span>Axial scan: '+str(len(result.z_m))+' planes · Δz '+plain_number(result.scan_step_m*1e3)+' mm</span>')
    return '<div style="font-size:13px;line-height:1.4;display:flex;flex-wrap:wrap;gap:3px 16px;align-items:baseline">'+''.join(items)+'</div>'


def _hardware_help():
    if xpos.select_backend('auto')=='cupy':
        return '<b>'+html.escape(_cuda_device_name())+'</b> · CuPy acceleration is active.'
    return '<b>CPU / NumPy</b> · To enable GPU in Colab: <b>Runtime → Change runtime type → GPU → Save</b>, then reconnect and run all cells. The actual CUDA device is checked.'


def _cuda_device_name():
    try:
        device=xpos.cp.cuda.Device()
        name=xpos.cp.cuda.runtime.getDeviceProperties(device.id)['name']
        if isinstance(name,bytes):name=name.decode()
        return 'CUDA GPU · '+str(name)
    except Exception:
        return 'CuPy / CUDA GPU'

def sync_results_notice():
    result=state.get('result')
    if result is None:
        results_notice.layout.display='none'
        return
    try:changed=read_configuration()!=result.configuration
    except (ValueError,TypeError):changed=True
    results_notice.value='<div class="xposim-notice">Parameters changed. Press Compute to update these results.</div>' if changed else ''
    results_notice.layout.display='' if changed else 'none'


REPORT_PAGE=(210/25.4,297/25.4)
REPORT_MARKERS=('#355f85','#147a70','#9b592d')


def report_value(value,scale=1,unit=''):
    if value is None or not math.isfinite(value):
        return '—'
    return f'{value/scale:.6g}'+(' '+unit if unit else '')


def report_values(values,scale=1,unit=''):
    text=[report_value(value,scale) for value in np.atleast_1d(values)]
    if len(text)==1 or text[0]==text[1]:
        combined=text[0]
    else:
        combined='x '+text[0]+' / y '+text[1]
    return combined+(' '+unit if unit and combined!='—' else '')


def report_planes(result):
    """Take a representative sampled plane on each side of the exact reference."""
    center=result.peak_z_m
    z=result.z_m
    before=z[z<center]
    after=z[z>center]
    reaches=[abs(float(value)-center) for value in (z[0],z[-1]) if value!=center]
    widths=[value for value in result.depth_of_focus_m if value is not None]
    offset=min(max(widths)*.5 if widths else float(np.ptp(z))*.25,min(reaches,default=0))
    def sample(options,target):
        return result.plane_at(float(options[np.argmin(abs(options-target))])) if options.size else None
    reference=xpos.CRLPlaneResult(center,result.best_intensity,result.x_m,result.y_m,
                                  result.reference_metrics,result.magnification_at_peak)
    return sample(before,center-offset),reference,sample(after,center+offset)


def report_axis(ax,*,grid=True):
    ax.tick_params(labelsize=7,length=2.5)
    for spine in ax.spines.values():
        spine.set_color('#a7b7c8')
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=3))
    ax.xaxis.set_major_formatter(TICK_FORMATTER)
    ax.yaxis.set_major_formatter(TICK_FORMATTER)
    if grid:ax.grid(color='#dbe4ee',linewidth=.55)
    ax.set_axisbelow(True)


def report_scan(fig,result,planes,reference):
    top,peak,width=[fig.add_axes([.112,y,.823,h]) for y,h in ((.503,.064),(.451,.047),(.395,.051))]
    z=result.z_m*1e3
    u=result.x_m*1e6
    top.pcolormesh(z,u,result.longitudinal_intensity.T/reference,shading='auto',
                   cmap=CMAP,norm=Normalize(0,1),rasterized=True)
    top.set_ylabel('x (µm)',fontsize=8)
    peak_scale,peak_unit=engineering(result.peak_intensity,'a.u.')
    peak.plot(z,result.peak_intensity/peak_scale,color='#355f85',lw=1.35)
    peak.set_ylabel('Peak I\n'+peak_unit,fontsize=7.2)
    widths=np.reshape(result.fwhm_m,(z.size,-1))
    for i,axis in enumerate(result.fwhm_axes):
        width.plot(z,widths[:,i]*1e9,color=('#356a57','#759c8c')[i],
                   lw=1.45,ls='--' if i else '-',label=axis)
    if len(result.fwhm_axes)>1:
        width.legend(loc='upper center',fontsize=6.6,ncol=2,frameon=True)
    width.set(xlabel='z from CRL exit (mm)',ylabel='FWHM (nm)')
    width.xaxis.label.set_fontsize(8);width.yaxis.label.set_fontsize(7.4)
    for ax in (top,peak,width):
        report_axis(ax,grid=ax is not top)
        ax.set_xlim(z[0],z[-1])
        for plane,color in zip(planes,REPORT_MARKERS):
            if plane is not None:
                ax.axvline(plane.z_m*1e3,color='white' if ax is top else color,
                           lw=.9,ls='-' if plane is planes[1] else ':',alpha=.9)
    for ax in (top,peak):ax.tick_params(labelbottom=False)


def report_transverse(fig,result,planes,reference):
    two_dimensional=result.y_m is not None
    pane_x=(.088,.374,.660)
    first_visible=next(i for i,plane in enumerate(planes) if plane is not None)
    if two_dimensional:
        x_um,y_um=result.x_m*1e6,result.y_m*1e6
        span=max(np.ptp(x_um),np.ptp(y_um))
        x_center=(x_um[0]+x_um[-1])/2
        y_center=(y_um[0]+y_um[-1])/2
    focus_labels=('A · BEFORE','B · NUMERICAL PEAK','C · AFTER')
    for i,(x,plane,label,color) in enumerate(zip(pane_x,planes,focus_labels,REPORT_MARKERS)):
        fig.text(x,.324,label,fontsize=8.2,weight='bold',color=color)
        ax=fig.add_axes([x,.123,.243,.172])
        if plane is None:
            ax.set_axis_off()
            ax.text(.5,.49,'Outside the sampled\nz range',ha='center',va='center',transform=ax.transAxes,
                    color='#526379',fontsize=8.5)
            continue
        note='z = '+report_value(plane.z_m,1e-3,'mm')
        fig.text(x,.306,note,fontsize=8,color='#283b50')
        if two_dimensional:
            ax.pcolormesh(plane.y_m*1e6,plane.x_m*1e6,plane.intensity.T/reference,
                           shading='auto',cmap=CMAP,norm=Normalize(0,1),rasterized=True)
            ax.set_aspect('equal',adjustable='box')
            if i==first_visible:ax.set_ylabel('x (µm)',fontsize=7,labelpad=1)
            else:ax.tick_params(labelleft=False)
        else:
            ax.plot(plane.x_m*1e6,plane.intensity/reference,color=color,lw=1.5)
            ax.set_ylim(-.02,1.035)
            if i==first_visible:ax.set_ylabel('I / Iref',fontsize=7,labelpad=1)
            else:ax.tick_params(labelleft=False)
        report_axis(ax,grid=not two_dimensional)
        if two_dimensional:
            ax.set_xlim(y_center-span/2,y_center+span/2)
            ax.set_ylim(x_center-span/2,x_center+span/2)
        else:
            ax.set_xlim(result.x_m[[0,-1]]*1e6)
        widths=report_values(np.atleast_1d(plane.metrics.fwhm)[:len(result.fwhm_axes)],1e-9,'nm')
        fig.text(x,.088,'FWHM '+widths,fontsize=7.4,color='#283b50')
    fig.text(.495,.103,'y (µm)' if two_dimensional else 'x (µm)',ha='center',fontsize=7)
    fig.text(.258,.071,'Shared map scale · I / Iref',fontsize=7.1,color='#526379')
    color_axes=fig.add_axes([.456,.071,.272,.007])
    colorbar=ColorbarBase(color_axes,cmap=CMAP,norm=Normalize(0,1),orientation='horizontal',
                         ticks=(0,.5,1))
    colorbar.ax.tick_params(labelsize=7,length=2,pad=1)
    colorbar.outline.set_visible(False)


def report_metrics(fig,result,analytical):
    cfg=result.configuration
    kind='diameter' if cfg.geometry=='two-dimensional' else 'width'
    magnification=analytical.magnification_at_image[:len(result.fwhm_axes)]
    rows=[
        ('Analytical focal distance',report_value(analytical.collimated_focus_m,1e-3,'mm')),
        ('Image-plane magnification',report_values(magnification)),
        ('Depth of focus',report_values(result.depth_of_focus_m,1e-3,'mm')),
        ('Peak intensity at B',report_value(result.reference_metrics.max_intensity,unit='a.u.')),
        ('Transmission',report_value(result.transmission,.01,'%')),
        ('Effective aperture '+kind,report_value(result.effective_aperture_m,1e-6,'µm')),
        ('Rocking FWHM',report_value(analytical.rocking_fwhm_rad,1e-3,'mrad')),
        ('Web transmission',report_value(analytical.web_transmission,.01,'%')),
        ('δ / β',report_value(analytical.delta)+' / '+report_value(analytical.beta)),
    ]
    if cfg.source_type=='gaussian':
        rows.append(('Gaussian / point FWHM',report_values(result.source_broadening_factor)))
    for i,(label,value) in enumerate(rows):
        label_x,value_x=(.066,.460) if i<5 else (.513,.933)
        y=.703-.021*(i%5)
        fig.text(label_x,y,label,fontsize=7.8,color='#43576e')
        fig.text(value_x,y,value,fontsize=8.0,ha='right',color='#13273e',weight='semibold')


def report_footer(fig,result):
    def mesh(shape,dx,dy):
        return (' × '.join(map(str,reversed(shape)))+' · Δx '+report_value(dx,1e-9,'nm')
                +(' · Δy '+report_value(dy,1e-9,'nm') if dy is not None else ''))
    z=result.z_m
    grid='Propagation '+mesh(result.grid_shape,result.grid_step_m,result.grid_step_y_m)
    propagation=(result.grid_shape,result.grid_step_m,result.grid_step_y_m)
    evaluation=(result.detector_shape,result.detector_step_m,result.detector_step_y_m)
    if propagation!=evaluation:grid+='    |    Field evaluation '+mesh(*evaluation)
    fig.text(.054,.045,grid,fontsize=7.2,color='#394d64')
    region='manual bounds' if result.configuration.scan_range_m is not None else 'auto bounds'
    same_peak=math.isclose(result.intensity_reference,result.reference_metrics.max_intensity,rel_tol=1e-9)
    ref_label='peak at B' if same_peak else report_value(result.intensity_reference,unit='a.u.')
    fig.text(.054,.031,f'Axial scan {z.size} planes · Δz {report_value(result.scan_step_m,1e-3,"mm")} · '
             f'{region}    |    Iref = {ref_label}',fontsize=7.2,color='#394d64')
    info_flags={'central_source_reference','convolution_model','mesh_window'}
    names={'detector_window':'evaluation window limit','detector_cap':'evaluation grid cap',
           'detector_boundary':'intensity reaches transverse boundary','sampling':'coarse FWHM sampling',
           'fwhm_missing':'unresolved FWHM','z_roi_boundary':'axial search boundary'}
    codes=[item.partition(':')[0] for item in result.diagnostics]
    diagnostics=[names.get(code,code.replace('_',' ')) for code in codes if code not in info_flags]
    if not result.numerical_peak.converged:diagnostics.append('peak search did not converge')
    if diagnostics:
        label='Flags: '+', '.join(dict.fromkeys(diagnostics))
        fig.text(.054,.017,textwrap.shorten(label,width=115,placeholder=' …'),fontsize=7,color='#8b512b')


def draw_report_qr(figure):
    from matplotlib.collections import PolyCollection

    # Fixed repository URL, QR version 3, error correction M.
    repository='https://github.com/mamichberdey/XPOSim'
    data=bytes.fromhex(
        'feba23fc17f5906eb9a6bb74e8b5dbadff2ec13d1507faaaafe00fb3009facd4be04'
        '73ad8bb358693bd9de9e34d0b0ea9b51fe8d17faad021635a7d29c446bb385b79854'
        '33d13724cf286cff0069c463fb8eeb105f5f10bade1fcdd48d206e94c36f04f046df'
        'e9f1a800'
    )
    size=29
    bits=np.unpackbits(np.frombuffer(data,dtype=np.uint8),count=size*size).reshape(size,size)
    cells=[[(x,y),(x+1,y),(x+1,y+1),(x,y+1)] for y,x in np.argwhere(bits)]
    ax=figure.add_axes([.857,.921,.090,.064])
    ax.add_collection(PolyCollection(cells,facecolors='black',edgecolors='none',antialiased=False))
    # Keep a four-module quiet zone around the code.
    ax.set(xlim=(-4,size+4),ylim=(size+4,-4),aspect='equal')
    ax.set_axis_off()
    figure.text(.902,.913,'GitHub',ha='center',fontsize=7.2,color='#355f85',url=repository)


def build_pdf_report(result,device,elapsed):
    """A single portrait A4 sheet tied to the last completed simulation."""
    from matplotlib.patches import Rectangle
    from matplotlib.lines import Line2D
    cfg=result.configuration
    analytical=xpos.analyze_crl(cfg)
    planes=report_planes(result)
    now=datetime.now(timezone.utc)
    basis='central-point reference' if cfg.source_type=='gaussian' else 'peak-intensity reference'
    if cfg.source_type=='gaussian':
        basis+=(' · Gaussian convolution' if cfg.source_method=='convolution'
                else f' · Gaussian point sampling ({cfg.source_points}/axis)')
    accuracy='Fast' if cfg.accuracy=='preview' else 'Accurate'
    backend=f'{device} · {elapsed:.2f} s'
    stream=io.BytesIO()
    style={'font.family':'DejaVu Sans','pdf.fonttype':42,'figure.facecolor':'white','axes.facecolor':'white',
           'text.color':'#172b42','axes.labelcolor':'#263a51','xtick.color':'#394d64','ytick.color':'#394d64',
           'savefig.bbox':None,'savefig.facecolor':'white'}
    with plt.rc_context(style):
        fig=plt.figure(figsize=REPORT_PAGE)
        try:
            fig.text(.054,.964,'CRL simulation report',fontsize=21,weight='bold')
            fig.text(.054,.940,f'{cfg.energy_kev:.6g} keV  ·  {accuracy}  ·  {basis}',fontsize=8.5,color='#3e5570')
            fig.text(.838,.962,f'XPOSim {xpos.__version__}  ·  {now:%Y-%m-%d %H:%M UTC}',ha='right',fontsize=7.8)
            fig.text(.838,.928,backend,ha='right',fontsize=7.8,color='#3e5570')
            draw_report_qr(fig)
            for y in (.906,.365,.057):
                fig.add_artist(Line2D([.054,.947],[y,y],transform=fig.transFigure,color='#bdccd9',lw=.7))
            fig.add_artist(Rectangle((.054,.605),.893,.137,transform=fig.transFigure,
                                    facecolor='#f0f5fa',edgecolor='#dbe5ee',lw=.75,zorder=-1))
            fig.text(.054,.900,'OPTICAL LAYOUT',fontsize=9,weight='bold')
            fig.text(.066,.722,'KEY MEASUREMENTS',fontsize=9,weight='bold')
            optical_schematic(cfg,analytical,ax=fig.add_axes([.054,.774,.481,.121]),show_depth=False)
            element_schematic(cfg,analytical,ax=fig.add_axes([.550,.760,.397,.137]),show_material=False)
            lens='Stack length '+report_value(analytical.lens_length_m,1e-3,'mm')
            if cfg.density_g_cm3 is not None:lens+='  ·  Density '+report_value(cfg.density_g_cm3,unit='g/cm³')
            fig.text(.054,.755,lens+'  ·  Max. N '+str(analytical.max_element_count),fontsize=7.4,color='#394d64')
            report_metrics(fig,result,analytical)
            fig.text(.054,.584,'PROPAGATION ALONG z',fontsize=9.5,weight='bold')
            fig.text(.054,.346,'TRANSVERSE SECTIONS',fontsize=9.5,weight='bold')
            reference=max(result.intensity_reference,np.finfo(float).tiny)
            report_scan(fig,result,planes,reference)
            report_transverse(fig,result,planes,reference)
            report_footer(fig,result)
            fig.savefig(stream,format='pdf',dpi=240,metadata={
                'Title':'CRL simulation report','Author':'XPOSim','CreationDate':now})
        finally:
            plt.close(fig)
    return stream.getvalue()


_report_directory=globals().get('_report_directory')

def download_report(_=None):
    global _report_directory
    result=state.get('result')
    if result is None or 'report_runtime' not in state:
        return
    report_button.disabled=True
    report_message.value='Preparing a print-ready A4 report…'
    try:
        data=build_pdf_report(result,*state['report_runtime'])
        filename='XPOSim_CRL_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')+'.pdf'
        # A self-contained link also works in notebook frontends without Colab's download API.
        encoded=base64.b64encode(data).decode('ascii')
        report_message.value=(f'<a download="{filename}" href="data:application/pdf;base64,{encoded}">'
                              'Save PDF report</a>')
        if colab_output is not None:
            from google.colab import files as colab_files
            if _report_directory is None:
                _report_directory=tempfile.TemporaryDirectory(prefix='xposim-report-')
            path=Path(_report_directory.name)/filename
            path.write_bytes(data)
            colab_files.download(str(path))
    except Exception as exc:
        if not report_message.value.startswith('<a '):
            report_message.value='PDF export failed · '+html.escape(str(exc))
    finally:
        report_button.disabled=state.get('result') is None


compute_button.on_click(compute_focus)
report_button.on_click(download_report)
hardware_notice=widgets.HTML('<div style="font-size:12px;line-height:1.25;padding:1px 0">'+_hardware_help()+'</div>')
left=widgets.VBox([scan_heading,scan_image,scan_compact_image],layout=widgets.Layout(flex='1 1 470px',min_width='0',width='auto'))
right=widgets.VBox([plane_heading,plane_image,plane_compact_image],layout=widgets.Layout(flex='1 1 470px',min_width='0',width='auto'))
left.add_class('xposim-panel');right.add_class('xposim-panel')
results_panel=widgets.HBox([left,right],layout=widgets.Layout(width='100%',flex_flow='row wrap',align_items='stretch',gap='10px',display='none'))
details.layout.display='none'
toolbar=widgets.HBox([compute_button,status,theme_selector],layout=widgets.Layout(width='100%',align_items='center',gap='12px',margin='3px 0 0'))
toolbar.add_class('xposim-toolbar')
result_css=UI_CSS.replace('</style>', '''.xp-depth-readout {font-size:12px;font-weight:500}
.xp-source-widths {display:flex;flex-wrap:wrap;align-items:baseline;gap:4px 14px;padding:6px 9px;
 border:1px solid var(--xp-border);background:var(--xp-surface);margin:0 0 6px;font-size:13px}
</style>''')
result_style=widgets.HTML(result_css,layout=widgets.Layout(height='0px',margin='0',overflow='visible'))
explorer_root=widgets.VBox([result_style,hardware_notice,toolbar,results_notice,progress_view,summary,source_comparison,results_panel,plane_controls,details,report_panel],
                         layout=widgets.Layout(width='100%',max_width='1200px',margin='0 auto'))
explorer_root.add_class('xposim-explorer')
results_panel.add_class('xposim-results')
right.add_class('xposim-detector')
plane_controls.add_class('xposim-slider-below')
plane_controls.add_class('xposim-selection-controls')
summary.add_class('xposim-summary')
theme_selector.observe(lambda change:apply_theme(),names='value')
display(explorer_root)
apply_theme(refresh=False)
if not _first_explorer_initialization:
    status.value='<span style="font-size:12px">Parameters retained. Press Compute.</span>'


<details>
<summary>How results are calculated</summary>

Here $D$ is the full aperture, $R$ the vertex radius, $t$ the bridge thickness, $N$ the element count and $s$ the source distance to the entrance. Scan coordinate $z$ is measured from the exit.

| Quantity | Expression and meaning |
| --- | --- |
| Element length and stack length | $p=t+D^2/(4R)$, $L=Np$. The diagram scales each dimension for readability. |
| Analytical focal and image distances | Set $L_c=\sqrt{pR/(2\delta)}$, $u=L/L_c$, $F=L_c/\sin u$, $h=F(1-\cos u)$. The parallel-beam focus is $F\cos u$. The finite-source image is $F(s+h)/(s+h-F)-h$. |
| Maximum CRL elements and magnification | The plane-wave focus is outside the CRL for $Np<\pi L_c/2$. Signed image magnification is $M=-B/s$, with $B$ from the entrance-to-image ray transfer matrix. |
| Numerical peak intensity distance | Search for a local maximum of $I_{\max}(z)=\max_{x,y}\lvert E(x,y,z)\rvert^2$, starting near the analytical image. For a Gaussian source this search uses the central point. Manual scan bounds limit the axial interval. |
| FWHM and depth of focus | $w=x_+-x_-$ at the two half-maximum crossings. The shaded axial band spans the crossings of $w(z)=\sqrt{2}\,w_{\min}$, around the sampled minimum on each focusing axis. Both crossings must be in the scan. |
| Transmission and effective aperture | $T=P_{\rm out}/P_{\rm in}$, with input power inside the entrance pupil and output power over the full exit. Effective width is $DT$. Effective diameter for a circular CRL is $D\sqrt{T}$. |
| Web transmission | $T_{\rm web}=\exp(-4\pi\beta Nt/\lambda)$ gives the transmission through the combined central bridge thickness. |
| Rocking curve FWHM | The analytical Gaussian-pupil model gives $T(\theta)/T(0)=\exp(-K\theta^2)$ and width $2\sqrt{\ln 2/K}$. $K$ describes the angle-dependent absorption of a tilted plane wave through the stack. This estimates the focusing-direction acceptance. The curve is flat at $K=0$. |
| Source broadening | $w_{\rm Gaussian}/w_{\rm point}$ compares the full-source and central-point FWHM in the same numerical reference plane. Point sampling adds $I=\sum_j w_j\lvert E_j\rvert^2$. Convolution uses shifted central-point patterns. |
| Relative intensity | Both maps show $I/I_{\rm ref}$, using the same peak from the full-source scan. The 2D longitudinal map takes the middle y row of the sampled region. |

For the angular estimate, $k=2\pi/\lambda$, $g=2k(\beta+i\delta)/R$ and $A=D$ for one-axis focusing or $A=D\sqrt{\pi}/2$ for circular focusing. Start at the entrance with $(a,b)=(\pi/A^2,ik)$. A drift $\ell$ maps $(a,b)$ to $(a,b)/(1+i\ell a/k)$. Drift by $p/2$ to the first element centre. At each centre, add

$$
\frac{\Re g}{\Re a+\Re g}\frac{(\Re b)^2}{\Re a}
$$

to $K$, update $a\leftarrow a+g$ and drift by $p$ to the next centre.

**Fast** uses propagation-grid samples. **Accurate** evaluates the complex field on a finer transverse evaluation grid. Tight calculation bounds can leave width crossings unresolved. [Rocking-curve model in Kohn (2018)](https://journals.iucr.org/s/issues/2018/06/00/mo5185/)

</details>
